<!-- Last changed: Facebook_20260825E -->
> **Enclave notebook — `Facebook_YYYYMMDD<A-Z>.ipynb` series.** This file is *authoring
> only*. It imports `metacontentlibraryapi`, which exists only inside Meta's Secure
> Research Environment; it cannot be executed, tested, or run against the API from
> this repo. Committed with **all outputs cleared** — Meta permits no data egress, so
> nothing produced by running it may be saved back here.
>
> Predecessors: `Facebook_20260824A.ipynb` (checkpointing), `Facebook_20260822A.ipynb`
> (aggregate grain), `Claude_Reference/H5N1_Aggregate_20260819.ipynb` (origin).

# H5N1 Farm-Page Post Aggregation — Meta Content Library

Page discovery, post collection, term flagging, sentiment, and a single aggregate table
at the grain **(PageCategory, State, Zip, Date)**.

Every code cell carries a `# BLOCK nn - name` header. Blocks run in order; the two
long-running ones (**07** page discovery, **12** post pull) print a line per slice or
job and are meant to be **collapsed** — each is followed immediately by a summary block
(**08**, **13**) that recomputes and prints everything you actually need to read.

---

## What the last full run measured — and why this revision exists

Page discovery on 2026-08-24, all eight categories:

| stage | pages |
|---|---|
| returned by `facebook/pages/preview` | 18,062 |
| US-located | 13,265 |
| with a wanted **primary** category | 12,787 |
| primary category `Farm` | 3,837 |

`facebook/posts/estimate` over those surfaces returned **~6,000,000 posts**. Two things
follow, and both are why this is a major revision rather than a patch:

**1. Every async job would have been silently truncated.** 12,787 surfaces is 52 batches
of 250. 6,000,000 ÷ 52 ≈ **115,000 posts per job against a documented 100,000 ceiling**.
Restricting to Farm pages does not save it either — 3,837 surfaces is 16 batches at
~113,000 each. The old design had no axis on which to subdivide further, so `capped`
would have fired on essentially every batch and `n_posts` would have been a floor of
unknown tightness.

> **The fix: `JOB_WINDOW`.** Jobs are now partitioned by **surface batch × time window**,
> not by surface batch alone. At the default `"year"` that is 52 × 5 = 260 jobs of
> ~23,000 posts each — comfortably under the cap. `"quarter"` (988 jobs) and `"month"`
> (2,860 jobs) exist for narrower slicing, at 18 h and 52 h of submission time
> respectively given the 1 query/minute async rate limit.

**2. 6,000,000 posts is twelve weeks of query budget.** The budget is 500,000 records
per rolling 7 days, shared across sync and async. A full-scope `all_posts` pull is not a
single sitting; it is a staged collection. Block 10 therefore keeps a **budget ledger**
in the checkpoint and stops submitting when the next job's estimate would exceed what
the rolling window has left. Re-run next week and it picks up where it stopped.

If twelve weeks is not acceptable, the scope levers, in the order I would pull them:

1. **Narrow the window.** `PostStartDate = "2024-08-01"` covers the outbreak period and
   cuts the pull by roughly half.
2. **`RESTRICT_TO_FARM_ONLY = True`.** ~1.8M posts. Loses the cross-category comparison
   that made `page_category` a grouping dimension in the first place.
3. **`POST_QUERY_MODE = "broad_terms"`.** Cheapest by far, but it destroys the true
   denominator — `n_posts` stops meaning "everything these pages posted", which is the
   measurement this notebook exists to produce. Pull this lever last.

## Sharding the collection across researchers

Twelve budget-weeks is a long time to hold a pull open. The work is therefore split
three ways on **`page_category`**, selected by the `SHARD` name in block 02.

`page_category` is not merely a convenient key — it is the **first grouping key of the
aggregate**, so shards are disjoint in the output and combining is a `concat` with no
re-aggregation. Splitting on state, date, or page-id ranges would put the same aggregate
cell in two people's results and force a merge that re-derives means; `mean_sentiment`
cannot be re-derived from totals, so that merge would silently lose it. A page lands in
exactly one shard because block 09 filters on *primary* category, even though raw
`PAGE_ROWS` overlap between researchers.

| shard | categories | pages (2026-08-24) | est. posts | jobs | submit |
|---|---|---|---|---|---|
| `nick` | Farm, Animal Shelter | 4,247 | ~1.99M | 85 | 1.5 h |
| `max` | Agricultural Service, Agricultural Cooperative | 4,523 | ~2.12M | 95 | 1.7 h |
| `waree` | Veterinarian, Agriculture | 3,932 | ~1.84M | 80 | 1.4 h |
| `smoke` | Zoo | 28 | ~13k | 1 | seconds |

Rebalance off the real per-shard estimates rather than these — 469 posts/page is a flat
average across all categories, and posting rates almost certainly differ by category.

> ⚠️ **Whether this actually buys anything depends on how the record budget is scoped.**
> If 500,000 records / 7 days is **per researcher**, three shards finish in ~4 weeks. If
> it is **per research project**, the shards share one pool and the split buys only
> submission wall-clock (~3x, since the 1 query/minute async limit is per user). Confirm
> which before treating the four-week figure as real.

### Combining

Block 22 does it, and works whether or not researchers share an enclave filesystem: it
reads `shard_<name>_exportagg.*` plus a manifest sidecar from `SHARD_DIR`, preferring
CSV (which survives output review) over pickle (which does not). Run it inside the
enclave on a shared workspace, or outside on three separately-approved exports.

The mechanical merge is trivial. The real risk is **configuration drift** — three
people, three environments, one dataset. Block 22 hard-fails when shards disagree on
`SENTIMENT_ENGINE`, `POST_QUERY_MODE`, `RUN_START`/`RUN_END`, `ENGAGEMENT_MIN`,
`JOB_WINDOW`, or `MIN_CELL_POSTS`. `SENTIMENT_ENGINE` is the dangerous one: if one
researcher gets `vaderSentiment` and another silently falls back to the inline lexicon,
every number still computes, every sanity check still passes, and `mean_sentiment` is
quietly incomparable across a third of the corpus.

### Smoke preset

`SHARD = "smoke"` runs the whole pipeline against **Zoo** — 28 pages, one surface batch,
seconds. Zoo is deliberately excluded from every production shard, so a smoke run can
never be mistaken for or contaminate real data. Block 05A then prints a parity report
covering the environment, the sentiment fingerprint, and a checkpoint round-trip.

> ⚠️ **The parity report contains no counts, deliberately.** Versions, engine name,
> fixed-string sentiment scores, and pass/fail booleans are not Meta data and can be
> shared over Slack or email. A row count *is* derived from Meta data. Do not paste
> "got 847 pages" anywhere — that is an egress decision, not a status update.

## Page discovery is a measured sample, not a census

`facebook/pages/preview` is synchronous-only — there is no `facebook/pages/job` — so
every individual search is capped at 1,000 results in **both** run modes. Discovery is
partitioned one search per category, then sub-split across `PageDescription` terms for
any category that still hits the cap.

**On the last run, 13 slices were still at the 1,000 cap after sub-splitting**, spanning
at least `Farm` and `Agricultural Service` across the terms *livestock, farm, farming,
ranch, agriculture, agricultural*. Those slices are truncated by an unknown amount. A
third level — `PAGE_REFINE_CAPPED`, which re-runs a capped slice as `(term & refiner)`
AND-groups — is available in block 07 and **defaults to off**, because widening the page
universe makes the post-budget problem strictly worse. Turn it on only if the page
universe itself is the object of study.

## Three weightings, because they answer three different questions

The aggregate carries the same underlying reactions three ways, and the difference
between them is the whole point rather than redundancy:

| statistic | weights by | what it answers |
|---|---|---|
| `like_ratio_total` | **reactions** | pooled over the cell. One national brand with 50,000 reactions dominates every small farm in the state. |
| `mean_like_ratio` | **posts** | each post counts once, however popular. |
| `mean_like_ratio_page_eq` | **pages** | each page counts once, however big. Ten small farms no longer collapse into one national brand. |

`max_sad_share_term` is not an average at all. It is the **loudest single distress signal
in the cell among posts that matched a real `PostText` concept** — the maximum of
`sad / (like + sad)` over term-matching posts. One small farm reporting a single dead
bird is the event of interest, and every mean above buries it by construction. The
term restriction is what makes it a bird-flu signal rather than a general-misery signal;
`n_term_ratio_defined` is the count it was taken over, and a max over one post means
something very different from a max over fifty.

`mean_sentiment_page_eq` applies the same page-equal weighting to sentiment. Sentiment is
the dependent variable, so the argument for it is if anything stronger than for the
ratio.

## No median, and why

`median_sentiment` was removed in `20260825E`. It was never a page-equal statistic —
that is `mean_sentiment_page_eq` — it was the median over *posts*, a central-tendency
measure whose entire purpose is resistance to outliers. In the ten-small-farms-and-one-
national-brand cell, the median is guaranteed to report the ordinary farm and ignore the
one reporting a dead bird. It answers the opposite of the question this notebook asks.

Two extremes replace it:

* **`min_sentiment_term`** — the most negative *term-matching* post in the cell. This is
  the exact sentiment twin of `max_sad_share_term`, and the two should be read together:
  one measures how sad the reactions were, the other how negative the text was.
* **`min_sentiment`** — the most negative post regardless of term. Useful as the control
  the term-restricted version is compared against: distress that never used a flu word is
  the baseline that `min_sentiment_term` has to beat to be evidence of anything.

## Model-ready columns: NaN kept, and filled

The term-restricted signals are NaN wherever no term-matching post qualified — either
because no post in the cell matched a `PostText` concept at all, or because the ones that
did got no reactions. Both are common, and both are honest: NaN means *not measurable
here*, which is not the same as zero.

But a downstream model cannot consume NaN without dropping the row, and those rows are
the quiet majority. So each signal is carried twice:

| column | meaning |
|---|---|
| `max_sad_share_term` | the measurement. NaN where nothing qualified. |
| `max_sad_share_term_model` | the same, with NaN filled by `MODEL_FILL_VALUE` (0.0). |

Same for `min_sentiment_term` / `min_sentiment_term_model`. Block 17 reports how many
cells were filled and, importantly, *why* — split by cause.

> **On filling with zero.** This project has a standing rule that suppressed NASS values
> must never be coalesced to 0, because withheld inventory means *unknown* and zeroing it
> is unrecoverably wrong. The rule holds; this is not a violation of it, and the
> difference is worth being explicit about. There, the zero destroys information and
> nothing marks the rows it touched. Here, `n_posts_term`, `n_term_ratio_defined`, and
> `n_term_sentiment_scored` identify the filled set exactly, the NaN column is kept
> alongside, and 0.0 carries the right meaning for these particular measures — "no
> observed distress", "no observed negativity". If any of those three conditions stopped
> being true, the fill would become the NASS mistake.

Other float columns (`mean_sentiment`, `like_ratio_total`, the page-equal means) are NaN
only when a cell had nothing to measure at all, which is rarer and is properly the
modelling notebook's decision rather than this one's. Block 17 prints the NaN rates so
that decision can be made with numbers.

## Thin cells are retained, deliberately

`MIN_CELL_POSTS` defaults to **0 — no suppression** — and that is now a considered
position rather than an unset knob. A cell with `n_posts = 1` is not noise to be tidied
away; under the reasoning above it is very often *the signal*. Suppressing at 5 or 10
would delete exactly the small single-farm observations that `max_sad_share_term` exists
to surface.

> ⚠️ **I am confabulating** on what Meta's output review actually requires. The lever is
> retained (`MIN_CELL_POSTS`) so a reviewer's requirement can be met without a code
> change, but whether thin cells must be suppressed in an export of *counts and ratios
> with no post text* is a question for Meta's review team, not something to infer. Ask
> before assuming either way. Nothing about retention changes inside the enclave — the
> analysis dataset always keeps every cell; suppression is a review-time decision applied
> only to what is requested out.

### A note on names

`SourcePosts` holds **every post from every page in this shard**, not farm posts. It was
called `FarmPosts` through `20260825C` — a leftover from `H5N1_20260819`, where
`PageCategories` really was `["Farm"]`. It stopped being true in `20260822A` when the
category list widened to make `page_category` a grouping dimension, and the name did not
follow. Renamed in `20260825D` because a name that asserts a filter the code does not
apply is worse than a vague one: a reviewer reading block 17 could reasonably assume the
aggregate is farm-scoped.

The lineage now reads straight through: `CleanUSPages` → `SourcePages` → `SourcePosts`.

`FarmPages` keeps its name — it genuinely is the Farm-only subset, and it feeds
`SourcePages` only when `RESTRICT_TO_FARM_ONLY` is set. `FarmPostsFiltered` became
`PostsWithTerm`, which is what it always was: the term-matching subset, retained for
continuity with the older notebooks and used nowhere downstream.

### Aggregate output columns
| Column | Meaning |
|---|---|
| `n_posts` | all posts in the cell |
| `n_posts_term` | posts whose text matches a `PostText` term |
| `n_engagement_ge10` | posts where `like_count + sad_count >= 10` |
| `n_ratio_defined` / `n_ratio_undefined` | posts with `like+sad > 0` / `== 0` |
| `n_sentiment_scored` | posts with text that scored |
| `like_count_total` / `sad_count_total` | summed reactions |
| `n_pages` | distinct pages contributing to the cell |
| `n_term_ratio_defined` | term-matching posts with a defined ratio |
| `like_ratio_total` | `like/(like+sad)` pooled over the cell — reaction-weighted |
| `mean_like_ratio` | mean of `like/(like+sad)` over `n_ratio_defined` posts — post-weighted |
| `mean_like_ratio_page_eq` | mean over pages, each page counting once — page-weighted |
| `mean_sentiment` | compound sentiment over `n_sentiment_scored` posts |
| `min_sentiment` | most negative post in the cell |
| `min_sentiment_term` | most negative **term-matching** post — the sentiment twin of `max_sad_share_term` |
| `n_term_sentiment_scored` | term-matching posts that scored |
| `max_sad_share_term_model` / `min_sentiment_term_model` | the two signals with NaN filled — what the model consumes |
| `mean_sentiment_page_eq` | mean sentiment over pages, each page counting once |
| `max_sad_share_term` | max `sad/(like+sad)` among term-matching posts — the early-warning signal |

### Four things to know before you trust the numbers
1. **`n_posts` is a true "every post these pages made" denominator in the default
   mode.** `POST_QUERY_MODE = "all_posts"` sends no `q`. If the API rejects a `q`-less
   query, switch to `"broad_terms"` and read `n_posts` as "broad-term posts," not
   "all posts."
2. **Zip and State are the *page's* registered location, not the author's.** Meta
   exposes no author geography. See the `meta-content-library-constraints` memory.
3. **Sentiment is lexicon-based and computed in-enclave.** No model weights are
   downloaded — the enclave has no egress. See block 05 for the fallback chain.
   **As of the 2026-08-25 smoke run the enclave resolves to `inline-lexicon`** —
   fallback #3, the hand-built domain lexicon in block 05, not VADER. That is fine for
   comparability as long as all three shards agree (block 05A's fingerprint proves it),
   but the methods write-up must not describe these scores as VADER scores.
4. **Nothing here leaves the enclave without Meta's output review.** Block 20 applies
   small-cell suppression to make the aggregate reviewable; raw post text is never part
   of the export shape. Checkpoints (blocks 04, 21) write to the enclave's own disk and
   are **not** an export.


In [ ]:
# BLOCK 00 - Imports and environment
# Last changed: Facebook_20260825A
from metacontentlibraryapi import MetaContentLibraryAPIClient as client
from datetime import date, timedelta
import gzip
import json
import math
import os
import re
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

print(f"pandas {pd.__version__}   numpy {np.__version__}")
print(f"cwd    {os.getcwd()}")


In [ ]:
# BLOCK 01 - Search vocabulary
# Last changed: Facebook_20260825B
# ----------------------------------------------------------------------------
# PAGE DISCOVERY (who is posting)
# ----------------------------------------------------------------------------
# PageCategories is NO LONGER SET HERE. It is derived from SHARD in block 02, so
# that three researchers cannot drift into three different category lists by
# hand-editing. The reference counts below are the 2026-08-24 discovery run and
# exist so block 09 can report drift against a known baseline.
PageCategoryPages = {
    "Farm": 3837,
    "Agricultural Service": 3340,
    "Veterinarian": 1996,
    "Agriculture": 1936,
    "Agricultural Cooperative": 1183,
    "Animal Shelter": 410,
    "Wildlife Sanctuary": 57,   # dropped from production: low return
    "Zoo": 28,                  # dropped from production; reserved for SHARD="smoke"
}

PageDescription = [
    "poultry", "chicken", "turkey", "egg", "hatchery",
    "flock", "livestock", "farm", "farming", "agriculture",
    "agricultural", "ranch", "dairy", "veterinary", "veterinarian",
    "avian",
]

# Third-level split for slices still at the 1,000 cap after sub-splitting by
# PageDescription. Applied as "(term & refiner)" AND-groups, so coverage is
# incomplete by construction: a page whose description carries the term but none
# of these refiners stays invisible. Off by default - see PAGE_REFINE_CAPPED.
PAGE_REFINERS = [
    "poultry", "chicken", "turkey", "duck", "egg", "eggs", "hatchery",
    "flock", "broiler", "layer", "pasture", "organic", "family", "local",
]

PageCountry_Code = "US"

# ----------------------------------------------------------------------------
# POST DISCOVERY (what they wrote)
# ----------------------------------------------------------------------------
# PostText = the *precise* concept list. Multi-word entries are phrases, which
# the API cannot search, so they are enforced client-side as has_post_term.
PostText = [
    "avian influenza", "avian flu", "chicken flu", "bird flu", "poultry flu",
    "zoonotic flu", "duck flu", "farm flu",
    "h5n1", "h5n8", "h7n9", "h9n2", "h10n3",
    "hpai",   # Highly Pathogenic Avian Influenza
    "lpai",   # Low Pathogenic Avian Influenza
]

# PostSingleWordText = the *recall* list sent to the API (single tokens only).
# Only reaches the API when POST_QUERY_MODE == "broad_terms".
PostSingleWordText = [
    "influenza", "flu",
    "h5n1", "h5n8", "h7n9", "h9n2", "h10n3",
    "hpai", "lpai",
]

PostStartDate = "2022-01-01"
PostEndDate   = "2026-07-31"

print(f"{len(PageDescription)} description terms, "
      f"{len(PAGE_REFINERS)} refiners")
print(f"{len(PostText)} precise concepts, {len(PostSingleWordText)} API recall tokens")
print("page categories come from SHARD (block 02)")
print(f"configured post window: {PostStartDate} .. {PostEndDate}")


In [ ]:
# BLOCK 02 - Run mode, budget, and checkpoint configuration
# Last changed: Facebook_20260825G
# ----------------------------------------------------------------------------
# IDENTITY
# ----------------------------------------------------------------------------
# ONE place. Block 04's checkpoint manifest and block 05A's parity report both
# read this. They used to hardcode it separately, which is why every manifest
# written by 20260825B/C/D claimed to be 20260825A - stale provenance that only
# shows up when someone tries to reconcile a shard months later.
#
# Collapsing three hardcodes into one constant did NOT fix that failure, it only
# moved it: 20260825F shipped with this still reading 20260825E, so that entire
# run checkpointed and reported itself as E. Block 03 now cross-checks the
# constant against the real filename where the environment exposes one. BUMP
# THIS ON EVERY COPY-FORWARD - it is the first edit, not the last.
NOTEBOOK_VERSION = "Facebook_20260825G"

# ----------------------------------------------------------------------------
# MODEL-READY FILL
# ----------------------------------------------------------------------------
# The term-restricted signals are NaN wherever no term-matching post qualified,
# which is honest but unusable downstream: the H5N1 model would drop every such
# row, and those rows are the quiet majority. Each gets a *_model companion
# filled with this value, while the measurement column stays NaN.
#
# 0.0 is defensible here and ONLY here: for max_sad_share_term it means "no
# observed distress", for min_sentiment_term "no observed negativity", and
# n_posts_term / n_term_ratio_defined / n_term_sentiment_scored recover exactly
# which rows were filled. That is what makes this different from coalescing a
# suppressed NASS value to 0, where the zero is unrecoverable and wrong.
MODEL_FILL_VALUE = 0.0

# ----------------------------------------------------------------------------
# SHARD - which slice of the collection this run owns
# ----------------------------------------------------------------------------
# Splitting on page_category is what makes the shards recombinable: it is the
# FIRST grouping key of the aggregate, so shards are disjoint in the output and
# block 22 can concat them without re-aggregating. Any other split key would put
# the same aggregate cell in two researchers' output, and mean_sentiment cannot
# be re-derived from totals.
#
# Set this ONE name. Everything else - categories, checkpoint directory, the
# manifest tag - follows from it, so three researchers cannot drift into three
# different category lists by hand-editing.
SHARD = "smoke"      # "nick" | "max" | "waree" | "smoke"

SHARDS = {
    # Counts are the 2026-08-24 8-category baseline, and they are FLOORS: the
    # 2026-08-25 nick run reached 4,246 Farm + 410 Animal Shelter = 4,656 with
    # only two categories searched. Widening discovery raises them further.
    "nick":  ["Farm", "Animal Shelter"],                        # 4,247 pages
    "max":   ["Agricultural Service", "Agricultural Cooperative"],  # 4,523 pages
    "waree": ["Veterinarian", "Agriculture"],                   # 3,932 pages
    # Zoo is excluded from every production shard, so a smoke run can never be
    # mistaken for or contaminate real data.
    "smoke": ["Zoo"],                                           # 28 pages
}
if SHARD not in SHARDS:
    raise ValueError(f"SHARD must be one of {sorted(SHARDS)}, got {SHARD!r}")

PageCategories = SHARDS[SHARD]
IS_SMOKE = SHARD == "smoke"

# Where combined shard files are read from and written to (block 20 / block 22).
SHARD_DIR = os.path.join(os.getcwd(), "h5n1_shards")

# ----------------------------------------------------------------------------
# RUN MODE - preview smoke test vs. the full pull
# ----------------------------------------------------------------------------
# These are two DIFFERENT API mechanisms, not one endpoint with two names:
#
#   "preview" -> SYNCHRONOUS search. client.get("facebook/posts/preview").
#                10 results/page, hard-capped at 1,000 results per search.
#                Seconds. Also truncates page discovery and the date window.
#
#   "full"    -> ASYNCHRONOUS job. client.post("facebook/posts/job").
#                Up to 100,000 results per job, minutes to days, rate-limited
#                to 1 submission/minute. Resumable: re-run to keep polling.
RUN_MODE = "preview"          # "preview" | "full"

# Documented endpoint paths. There is no "facebook/posts/search".
PAGES_PREVIEW_PATH   = "facebook/pages/preview"    # sync only - no async variant
POSTS_PREVIEW_PATH   = "facebook/posts/preview"    # sync,  <= 1,000 results
POSTS_JOB_PATH       = "facebook/posts/job"        # async, <= 100,000 results
POSTS_ESTIMATE_PATH  = "facebook/posts/estimate"   # rough result-size estimate

# --- page discovery ---------------------------------------------------------
PARTITION_PAGE_SEARCH = True   # False = one unpartitioned search, capped at 1,000
PAGE_SEARCH_SLEEP_S   = 1.05   # sync limit is 60 queries/min; each paged call counts

# facebook/pages/preview returns HTTP 500 intermittently. On 2026-08-25 four Farm
# slices - poultry, chicken, egg, dairy - all failed that way and contributed ZERO
# pages, which is most of why that run came back 4.7% short of its baseline. Those
# are the four most poultry-relevant terms in the list, so losing them is a
# systematic bias, not a rounding error. Retry, and retry loudly.
PAGE_SEARCH_RETRIES     = 6    # attempts per API call, not per slice
PAGE_SEARCH_RETRY_SLEEP = 5.0  # seconds, doubling each attempt

# Re-run only the slices that errored, keeping everything already discovered.
# Without this a single 500 means redoing hours of discovery from scratch.
RETRY_FAILED_SLICES = True

# Farm/poultry and Farm/chicken failed on BOTH 2026-08-25 runs, the second time
# through four retries with backoff. Two consecutive whole-slice failures on the
# same two queries is not a transient 500 - it behaves like a query the server
# cannot complete. So when a term slice fails, ask for a strictly SMALLER result
# set rather than asking again: re-query it as "(term & refiner)" groups.
# Coverage is incomplete by construction - a page carrying the term and none of
# the refiners stays invisible - but incomplete beats the zero rows it returns
# now, and those are the two most poultry-relevant terms in the list.
PAGE_NARROW_FAILED = True

# Discovery runs across ALL production categories, not just this shard's.
# The API's `categories` filter is not an exact-primary match and every slice is
# capped at 1,000, so a Farm-primary page can be reachable only through some
# other category's slice. Restricting discovery to the shard's own categories
# therefore finds FEWER of the shard's own pages - that is the second reason the
# 2026-08-25 Farm count came in under baseline. Block 09 still filters to the
# shard, so the shards stay disjoint; only discovery is widened.
DISCOVER_ALL_CATEGORIES = True

# Third level of splitting for slices still capped after the PageDescription
# sub-split. 13 slices were still capped on the 2026-08-24 run. Left OFF because
# a wider page universe makes the post budget problem strictly worse.
PAGE_REFINE_CAPPED = False

# --- preview-mode throttles -------------------------------------------------
PREVIEW_MAX_SURFACES  = 250    # pages fed to the post pull (= one SURFACE_BATCH)
PREVIEW_MAX_API_PAGES = 25     # 25 pages x 10 results = up to 250 pages discovered
PREVIEW_WINDOW_DAYS   = 60     # trailing days of the configured post window

# --- full-mode (async job) settings ----------------------------------------
# SNAPSHOT: results retrievable for up to a year, periodically re-redacted.
# LIVE:     available for 30 days at most. SNAPSHOT is the reproducible choice
#           for a capstone that has to be re-run months later.
JOB_MODE            = "SNAPSHOT"
JOB_NAME            = "H5N1 farm-page posts"
JOB_DESCRIPTION     = ("H5N1 early-warning capstone: posts from US agriculture / "
                       "veterinary / animal-facility Pages.")
JOB_SUBMIT_SLEEP_S  = 65       # async rate limit is 1 query/min - stay under it
JOB_POLL_INTERVAL_S = 60       # seconds between status checks
JOB_POLL_BUDGET_MIN = 30       # one poll round; POLL_ROUNDS rounds per run
JOB_RESULT_CAP      = 100_000  # documented async ceiling

# THE 2026-08-25 FIX. One job per surface batch over the whole window averaged
# ~115,000 posts against a 100,000 ceiling - silent truncation on nearly every
# batch. Jobs are now (surface batch x time window).
#   None      -> one window, the old behaviour. Do not use at full scope.
#   "year"    ->   5 windows,  260 jobs, ~23,000 posts each,  4.7 h to submit
#   "quarter" ->  19 windows,  988 jobs,  ~6,100 posts each, 17.8 h to submit
#   "month"   ->  55 windows, 2860 jobs,  ~2,100 posts each, 51.6 h to submit
JOB_WINDOW = "year"

# --- query budget -----------------------------------------------------------
# 500,000 records per ROLLING 7 DAYS, shared by sync and async. The last
# estimate was ~6,000,000 posts: twelve budget-weeks. Submission stops when the
# next job's estimate would overrun what the rolling window has left, and the
# ledger is checkpointed so a later session resumes instead of restarting.
RECORD_BUDGET     = 500_000
MAX_JOBS_PER_RUN  = 60         # also caps submit wall-clock (x JOB_SUBMIT_SLEEP_S)

# Estimate result size before spending budget (full mode only).
RUN_ESTIMATE = True

# The estimate is a GATE, not a printout. While RUN_ESTIMATE is on and this
# stays False, block 12 prints the plan and STOPS without submitting anything.
CONFIRM_SUBMIT_JOBS = False

# ----------------------------------------------------------------------------
# OTHER TOGGLES
# ----------------------------------------------------------------------------
# "all_posts"   -> no q at all: n_posts is a TRUE denominator. This is the default.
# "broad_terms" -> q = PostSingleWordText (recall-filtered denominator, cheaper).
POST_QUERY_MODE = "all_posts"

RESTRICT_TO_FARM_ONLY = False  # True -> ~3,837 surfaces, ~1.8M posts
ENGAGEMENT_MIN        = 10     # reliability threshold for like_count + sad_count
MIN_CELL_POSTS        = 0      # small-cell suppression on the export table

# ----------------------------------------------------------------------------
# CHECKPOINTING - surviving a session timeout
# ----------------------------------------------------------------------------
# Written to the enclave's OWN disk. This is not an export: nothing crosses the
# Meta boundary, and no checkpoint substitutes for output review.
# Per-shard, so two shards run in one workspace cannot overwrite each other.
CKPT_DIR = os.path.join(os.getcwd(), f"h5n1_ckpt_{SHARD}")

# True  -> a re-run keeps in-memory state, else reloads it from CKPT_DIR.
# False -> ignore checkpoints and start clean, which RE-SPENDS query budget.
RESTORE_FROM_CHECKPOINT = True

# Poll rounds of JOB_POLL_BUDGET_MIN each, checkpointing after every one.
POLL_ROUNDS = 24

In [ ]:
# BLOCK 03 - Resolve run mode and job windows
# Last changed: Facebook_20260825G
# Everything downstream reads the resolved names below, never the raw config.
if RUN_MODE not in ("preview", "full"):
    raise ValueError(f"RUN_MODE must be 'preview' or 'full', got {RUN_MODE!r}")
if POST_QUERY_MODE not in ("all_posts", "broad_terms"):
    raise ValueError("POST_QUERY_MODE must be 'all_posts' or 'broad_terms', "
                     f"got {POST_QUERY_MODE!r}")
if JOB_MODE not in ("SNAPSHOT", "LIVE"):
    raise ValueError(f"JOB_MODE must be 'SNAPSHOT' or 'LIVE', got {JOB_MODE!r}")
if JOB_WINDOW not in (None, "year", "quarter", "month"):
    raise ValueError(f"JOB_WINDOW must be None/'year'/'quarter'/'month', got {JOB_WINDOW!r}")

# `until` must be strictly in the past or the API returns 400 / subcode 3790079.
_yesterday = (date.today() - timedelta(days=1)).isoformat()
if PostEndDate >= date.today().isoformat():
    print(f"PostEndDate {PostEndDate} is not in the past -> clamped to {_yesterday}")
    PostEndDate = _yesterday

# The smoke preset is applied HERE rather than trusted from block 02, so a
# leftover CONFIRM_SUBMIT_JOBS = True from a previous session cannot turn an
# environment check into a budget-spending run.
if IS_SMOKE:
    _forced = {"RUN_MODE": "preview", "RUN_ESTIMATE": False,
               "CONFIRM_SUBMIT_JOBS": False, "RESTRICT_TO_FARM_ONLY": False,
               "MIN_CELL_POSTS": 0}
    for _k, _v in _forced.items():
        if globals()[_k] != _v:
            print(f"SHARD='smoke': forcing {_k} = {_v!r} (was {globals()[_k]!r})")
        globals()[_k] = _v

# Version self-check. NOTEBOOK_VERSION is hand-set in block 02 and the
# copy-forward step forgets it - 20260825F ran and checkpointed itself as
# "Facebook_20260825E". Nothing downstream could detect that, so every manifest
# and parity report from that run carries the wrong version. Compare it against
# the real filename where the environment exposes one. WARN, never raise: the
# env var is not guaranteed to be set, and a wrong label must not kill a run
# that is holding hours of collection.
_nb_path = os.environ.get("JPY_SESSION_NAME", "")
_nb_stem = os.path.splitext(os.path.basename(_nb_path))[0] if _nb_path else ""
if _nb_stem and _nb_stem != NOTEBOOK_VERSION:
    print(f"!! NOTEBOOK_VERSION says {NOTEBOOK_VERSION!r} but this file is "
          f"{_nb_stem!r}.\n"
          f"   Fix block 02 and re-run blocks 02-03. The checkpoint manifest "
          f"and block 05A\n   read it, and a mislabelled checkpoint cannot be "
          f"reconciled later.")
elif not _nb_stem:
    print(f"note: filename not visible here, so NOTEBOOK_VERSION "
          f"{NOTEBOOK_VERSION!r} is\n      UNVERIFIED - confirm it matches the "
          f"file you actually opened.")

IS_PREVIEW = RUN_MODE == "preview"

# Discovery scope vs. shard scope. These are deliberately different: block 07
# searches DiscoveryCategories, block 09 keeps only PageCategories.
ALL_PRODUCTION_CATEGORIES = sorted({c for k, v in SHARDS.items() if k != "smoke"
                                    for c in v})
DiscoveryCategories = (ALL_PRODUCTION_CATEGORIES
                       if DISCOVER_ALL_CATEGORIES and not IS_SMOKE
                       else PageCategories)

MAX_SURFACES  = PREVIEW_MAX_SURFACES  if IS_PREVIEW else None
MAX_API_PAGES = PREVIEW_MAX_API_PAGES if IS_PREVIEW else None

RUN_END = PostEndDate
if IS_PREVIEW:
    _pv_start = date.fromisoformat(PostEndDate) - timedelta(days=PREVIEW_WINDOW_DAYS - 1)
    RUN_START = max(PostStartDate, _pv_start.isoformat())
else:
    RUN_START = PostStartDate


def _calendar_windows(start, end, grain):
    """Inclusive [since, until] pairs on calendar boundaries.

    Calendar-aligned rather than fixed-width so a re-run with a longer window
    reuses the earlier jobs' keys instead of shifting every boundary.
    """
    if grain is None:
        return [(start, end)]

    out = []
    cur = date.fromisoformat(start)
    last = date.fromisoformat(end)
    while cur <= last:
        if grain == "year":
            nxt = date(cur.year + 1, 1, 1)
        elif grain == "quarter":
            _q = (cur.month - 1) // 3
            nxt = (date(cur.year + 1, 1, 1) if _q == 3
                   else date(cur.year, 3 * (_q + 1) + 1, 1))
        else:  # month
            nxt = (date(cur.year + 1, 1, 1) if cur.month == 12
                   else date(cur.year, cur.month + 1, 1))
        stop = min(nxt - timedelta(days=1), last)
        out.append((cur.isoformat(), stop.isoformat()))
        cur = stop + timedelta(days=1)
    return out


# Preview is a single synchronous search: windowing does not apply.
JOB_WINDOWS = ([(RUN_START, RUN_END)] if IS_PREVIEW
               else _calendar_windows(RUN_START, RUN_END, JOB_WINDOW))

print(f"shard      : {SHARD}   categories {PageCategories}")
print(f"discovery  : {len(DiscoveryCategories)} categories "
      f"{'(all production - widens the page universe)' if DISCOVER_ALL_CATEGORIES and not IS_SMOKE else '(shard only)'}")
print(f"run mode   : {RUN_MODE}"
      + ("   <-- SMOKE TEST, counts are not the real dataset" if IS_PREVIEW
         else f"   <-- async jobs, {JOB_MODE} mode, minutes to days"))
# "preview" in the path below is the ENDPOINT'S name, not this run's RUN_MODE.
# There is no facebook/pages/job, so page discovery is synchronous and capped at
# 1,000 results per search whether RUN_MODE is "preview" or "full".
print(f"pages      : {PAGES_PREVIEW_PATH}"
      f"  (sync-only endpoint - no async variant exists;"
      f" 1,000/search in BOTH run modes)")
print(f"posts      : {POSTS_PREVIEW_PATH if IS_PREVIEW else POSTS_JOB_PATH}"
      f"  ({'sync, 1,000/search' if IS_PREVIEW else 'async, 100,000/job'})")
print(f"post window: {RUN_START} .. {RUN_END}"
      + (f"   (configured: {PostStartDate} .. {PostEndDate})" if IS_PREVIEW else ""))
print(f"surfaces   : {'all' if MAX_SURFACES is None else f'first {MAX_SURFACES}'}")
print(f"query mode : {POST_QUERY_MODE}")
print(f"checkpoints: {CKPT_DIR}")
if IS_SMOKE:
    print("\nSMOKE RUN - environment check only. Not a dataset, and it spends no\n"
          "result budget. Run block 05A and send Nick the parity report.")

if not IS_PREVIEW:
    print(f"job windows: {len(JOB_WINDOWS)} x {JOB_WINDOW}"
          f"   {JOB_WINDOWS[0][0]} .. {JOB_WINDOWS[-1][1]}")
    if JOB_WINDOW is None:
        print("  !! JOB_WINDOW=None at full scope averaged ~115,000 posts/job on the "
              "last run,\n     against a 100,000 ceiling. Expect silent truncation.")

<!-- Last changed: Facebook_20260825A -->
## 0 — Checkpoint machinery

An async pull outlives the session that started it. `JOB_MODE = "SNAPSHOT"` means Meta
holds the results for roughly a year, so the thing that must survive a session timeout
is the **job ID**, not the rows: lose `POST_JOBS` and you resubmit, re-spending budget
you have already paid. Lose the harvested rows and you re-harvest for free.

* `_ckpt_write` / `_ckpt_read` — one object to one file. Raw API rows go to gzipped
  JSONL so no `json_normalize` assumption is baked into the file; DataFrames go to
  gzipped pickle; small dicts and lists go to plain JSON.
* `_ckpt_init(NAME, default)` — in-memory state, else the checkpoint, else a fresh
  default. This is what makes a re-run cheap and a resumed session possible.
* `checkpoint(tag)` — persist every piece of run state that currently exists.

`_ckpt_write` never raises. A checkpoint that kills a twelve-hour run is worse than no
checkpoint at all.

**Resuming a lost session:** run blocks 00–04 with `RESTORE_FROM_CHECKPOINT = True`,
then run down the notebook. Page discovery skips, submitted jobs are not resubmitted,
polling reattaches, and the budget ledger carries its rolling window forward.

> ⚠️ A checkpoint is **not** an export. These files never leave the enclave, and writing
> one is not a substitute for Meta's output review. Same boundary as
> `job.write_data_to_file()`: a within-enclave convenience, nothing more.


In [ ]:
# BLOCK 04 - Checkpoint machinery
# Last changed: Facebook_20260825E
os.makedirs(CKPT_DIR, exist_ok=True)

# Persisted as gzipped JSONL: raw API payloads, kept in their original shape so a
# reload is not hostage to how json_normalize behaved that day.
_CKPT_ROWSETS = {"page_rows", "post_rows"}

# Persist order: cheapest and most irreplaceable first. POST_JOBS is a few hundred
# bytes and is the only handle on SNAPSHOT results Meta holds for a year; the
# DataFrames at the end are all recomputable from POST_ROWS.
_CKPT_STATE = ("POST_JOBS", "POST_ESTIMATES", "BUDGET_LEDGER", "PAGE_SLICES",
               "POST_BATCH", "PAGE_ROWS", "POST_ROWS",
               "SourcePages", "CleanUSPages", "SourcePosts",
               "AggByCatStateZipDate", "ExportAgg", "ExportAggState")

# Names that changed between notebook versions. A rename must not orphan a
# checkpoint written by an older run - that would quietly discard hours of
# collection. SourcePosts was called FarmPosts through 20260825C.
_CKPT_ALIASES = {"sourceposts": ("farmposts",)}


def _ckpt_write(name, obj):
    """Best-effort persist. NEVER raises - a checkpoint must not kill a long run."""
    try:
        if name in _CKPT_ROWSETS:
            p = os.path.join(CKPT_DIR, name + ".jsonl.gz")
            with gzip.open(p, "wt", encoding="utf-8") as fh:
                for r in obj:
                    fh.write(json.dumps(r, default=str) + "\n")
        elif isinstance(obj, pd.DataFrame):
            p = os.path.join(CKPT_DIR, name + ".pkl.gz")
            obj.to_pickle(p, compression="gzip")
        else:
            p = os.path.join(CKPT_DIR, name + ".json")
            with open(p, "w", encoding="utf-8") as fh:
                json.dump(obj, fh, default=str, indent=2)
        return p
    except Exception as e:
        print(f"  !! checkpoint FAILED for {name}: {type(e).__name__}: {e}")
        return None


def _ckpt_read(name):
    """Load one checkpointed object, or None if it is absent or unreadable.

    Falls back to any legacy name in _CKPT_ALIASES, so a checkpoint written
    before a rename still loads.
    """
    for candidate in (name,) + _CKPT_ALIASES.get(name, ()):
        for ext in (".jsonl.gz", ".pkl.gz", ".json"):
            p = os.path.join(CKPT_DIR, candidate + ext)
            if not os.path.exists(p):
                continue
            try:
                if ext == ".jsonl.gz":
                    with gzip.open(p, "rt", encoding="utf-8") as fh:
                        obj = [json.loads(ln) for ln in fh if ln.strip()]
                elif ext == ".pkl.gz":
                    obj = pd.read_pickle(p, compression="gzip")
                else:
                    with open(p, encoding="utf-8") as fh:
                        obj = json.load(fh)
                if candidate != name:
                    print(f"  note: loaded legacy checkpoint '{candidate}{ext}' "
                          f"for '{name}'")
                return obj
            except Exception as e:
                print(f"  !! checkpoint UNREADABLE {candidate}{ext}: "
                      f"{type(e).__name__}: {e}")
    return None


def _ckpt_init(varname, default):
    """In-memory state, else the checkpoint, else a fresh default.

    Re-running a cell must not re-fetch what has already been paid for, and a
    fresh kernel must be able to pick up where the dead one stopped.

    Job keys are STRINGS ("b003|2024-01-01") precisely so this survives a JSON
    round-trip - JSON has no integer keys, and the old int-keyed POST_JOBS had
    to be coerced back by hand on every restore.
    """
    if not RESTORE_FROM_CHECKPOINT:
        return default
    live = globals().get(varname)
    if live is not None and len(live):
        print(f"  {varname:<14} kept in memory      ({len(live):,})")
        return live
    disk = _ckpt_read(varname.lower())
    if disk is not None and len(disk):
        print(f"  {varname:<14} restored from disk  ({len(disk):,})")
        return disk
    return default


def _ckpt_manifest(tag):
    """Provenance. A checkpoint you cannot attribute to a run config is noise."""
    return {
        "tag": tag,
        "written_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "shard": globals().get("SHARD"),
        "notebook": NOTEBOOK_VERSION,
        "run_mode": globals().get("RUN_MODE"),
        "job_mode": globals().get("JOB_MODE"),
        "job_window": globals().get("JOB_WINDOW"),
        "run_start": str(globals().get("RUN_START")),
        "run_end": str(globals().get("RUN_END")),
        "query_mode": globals().get("POST_QUERY_MODE"),
        "page_categories": globals().get("PageCategories"),
        "sentiment_engine": globals().get("SENTIMENT_ENGINE"),
        "farm_only": globals().get("RESTRICT_TO_FARM_ONLY"),
        "engagement_min": globals().get("ENGAGEMENT_MIN"),
        "n_surfaces": len(globals().get("SourcePages", [])),
        "submit_gated": globals().get("SUBMIT_GATED"),
        "all_jobs_done": globals().get("ALL_JOBS_DONE"),
    }


def checkpoint(tag=""):
    """Persist whatever run state exists right now. Safe to call at any point."""
    _ckpt_write("run_manifest", _ckpt_manifest(tag))
    total = 0
    for name in _CKPT_STATE:
        obj = globals().get(name)
        if obj is None or not len(obj):
            continue
        p = _ckpt_write(name.lower(), obj)
        if p:
            total += os.path.getsize(p)
            print(f"  {name:<22} {len(obj):>9,}  ->  {os.path.basename(p)}")
    print(f"  checkpoint '{tag}' -> {CKPT_DIR}  ({total / 1e6:.1f} MB)")


print(f"shard          : {globals().get('SHARD')}")
print(f"checkpoint dir : {CKPT_DIR}")
print(f"restore        : {RESTORE_FROM_CHECKPOINT}")
_existing = sorted(os.listdir(CKPT_DIR))
print(f"on disk        : {_existing if _existing else 'nothing yet'}")


<!-- Last changed: Facebook_20260825E -->
## 1 — Sentiment engine

The enclave has no outbound network, so a `transformers` pipeline that pulls weights
from the Hub will fail. The chain below degrades gracefully:

1. `vaderSentiment` (pip package, lexicon ships inside the wheel) — preferred.
2. `nltk` VADER — only if `vader_lexicon` is already present on disk.
3. **Inline lexicon** — defined in the block, always works, zero dependencies.

All three return a compound score in `[-1, +1]` using VADER's normalization, so
`mean_sentiment` and the sentiment extremes are comparable whichever engine wins. Record
`SENTIMENT_ENGINE` in the methods write-up — a run scored by fallback #3 is not
interchangeable with one scored by #1.

Emoji are deliberately **not** scored: the prior cohort found emoji added noise rather
than signal.


In [ ]:
# BLOCK 05 - Sentiment engine
# Last changed: Facebook_20260825A
# ---------------------------------------------------------------------------
# Inline fallback lexicon. Small, domain-tilted, VADER-style valences on [-4, +4].
# ---------------------------------------------------------------------------
_LEX = {
    # --- outbreak / loss (domain-specific negatives) ---
    "cull": -2.6, "culled": -2.8, "culling": -2.8, "depopulate": -3.0,
    "depopulated": -3.0, "depopulation": -3.0, "euthanize": -2.8,
    "euthanized": -2.8, "euthanasia": -2.8, "outbreak": -2.6, "outbreaks": -2.6,
    "infected": -2.4, "infection": -2.3, "infectious": -1.8, "contagious": -1.8,
    "quarantine": -2.0, "quarantined": -2.0, "lockdown": -1.8, "biosecurity": -0.4,
    "dead": -2.9, "death": -2.9, "deaths": -2.9, "died": -2.8, "dying": -2.8,
    "mortality": -2.5, "carcass": -2.2, "carcasses": -2.2, "sick": -2.2,
    "illness": -2.0, "disease": -2.0, "diseased": -2.4, "virus": -1.6,
    "viral": -1.0, "epidemic": -2.8, "pandemic": -2.8, "spread": -1.0,
    "spreading": -1.2, "positive": -0.5, "confirmed": -0.6, "detected": -0.8,
    "detection": -0.8, "loss": -2.4, "losses": -2.4,
    "lost": -2.2, "devastating": -3.4, "devastated": -3.3, "tragedy": -3.2,
    "heartbreaking": -3.3, "heartbroken": -3.3, "crisis": -2.8, "emergency": -2.4,
    "shortage": -2.0, "recall": -1.6, "banned": -1.8, "ban": -1.6,
    "restriction": -1.4, "restrictions": -1.4, "closed": -1.4, "shut": -1.4,
    "struggle": -2.2, "struggling": -2.2, "suffering": -2.8, "risk": -1.4,
    "danger": -2.4, "dangerous": -2.6, "threat": -2.4, "fear": -2.4,
    "afraid": -2.2, "scared": -2.4, "worried": -2.2, "worry": -2.0,
    "concern": -1.4, "concerned": -1.6, "panic": -2.8, "anxious": -2.0,
    # --- general negatives ---
    "bad": -2.5, "awful": -3.0, "terrible": -3.1, "horrible": -3.2,
    "worst": -3.4, "sad": -2.6, "angry": -2.6, "hate": -3.0, "hard": -1.2,
    "difficult": -1.6, "problem": -1.8, "problems": -1.8, "fail": -2.4,
    "failed": -2.4, "failure": -2.6, "damage": -2.2, "destroy": -3.0,
    "destroyed": -3.0, "sorry": -1.2, "unfortunately": -1.8, "never": -1.0,
    # --- general positives ---
    "good": 1.9, "great": 3.1, "excellent": 3.2, "amazing": 3.2,
    "wonderful": 3.1, "fantastic": 3.2, "best": 3.2, "love": 3.2,
    "loved": 2.9, "happy": 2.7, "happiness": 2.8, "joy": 2.8, "excited": 2.4,
    "exciting": 2.4, "beautiful": 2.8, "proud": 2.4, "grateful": 2.6,
    "thank": 2.2, "thanks": 2.2, "thankful": 2.5, "blessed": 2.6,
    "welcome": 1.8, "enjoy": 2.2, "enjoyed": 2.2, "fun": 2.3, "nice": 1.8,
    "safe": 1.6, "safety": 1.0, "healthy": 2.2, "health": 0.8, "recovered": 2.2,
    "recovery": 1.8, "improving": 1.8, "improved": 2.0, "success": 2.7,
    "successful": 2.7, "support": 1.8, "supporting": 1.8, "help": 1.6,
    "helping": 1.6, "hope": 1.9, "hopeful": 2.2, "fresh": 1.6, "delicious": 2.6,
    "open": 1.0, "available": 0.8, "new": 0.6, "clean": 1.4, "strong": 1.8,
    "protect": 1.2, "protected": 1.6, "prevent": 0.8, "vaccine": 0.6,
    "vaccinated": 1.0,
}

_NEGATIONS = {
    "not", "no", "never", "none", "nobody", "nothing", "neither", "nowhere",
    "cannot", "cant", "wont", "dont", "doesnt", "didnt", "isnt", "arent",
    "wasnt", "werent", "hasnt", "havent", "hadnt", "aint", "without",
}
_BOOSTERS = {
    "very": 1.45, "really": 1.4, "extremely": 1.6, "so": 1.3, "absolutely": 1.5,
    "completely": 1.5, "totally": 1.45, "incredibly": 1.55, "super": 1.4,
    "highly": 1.4, "massively": 1.5, "entire": 1.3, "whole": 1.25,
    "slightly": 0.7, "somewhat": 0.75, "kinda": 0.7, "barely": 0.6,
    "hardly": 0.6, "little": 0.75, "sort": 0.8, "marginally": 0.7,
}
_TOKEN_RE = re.compile(r"[a-z0-9']+")


def _inline_compound(text):
    """VADER-shaped compound score on [-1, 1] from the inline lexicon."""
    if not text:
        return np.nan
    toks = _TOKEN_RE.findall(str(text).lower())
    if not toks:
        return np.nan
    total, found = 0.0, 0
    for i, tok in enumerate(toks):
        val = _LEX.get(tok)
        if val is None:
            continue
        found += 1
        # intensifier immediately before
        if i > 0 and toks[i - 1] in _BOOSTERS:
            val *= _BOOSTERS[toks[i - 1]]
        # negation within the preceding 3 tokens
        if any(t in _NEGATIONS for t in toks[max(0, i - 3):i]):
            val *= -0.74
        total += val
    if found == 0:
        return 0.0
    return float(np.clip(total / math.sqrt(total * total + 15.0), -1.0, 1.0))


# ---------------------------------------------------------------------------
# Engine selection
# ---------------------------------------------------------------------------
SENTIMENT_ENGINE = None
_scorer = None

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _SIA
    _an = _SIA()
    _an.polarity_scores("a good test")            # prove it runs
    _scorer = lambda t: _an.polarity_scores(str(t))["compound"] if t else np.nan
    SENTIMENT_ENGINE = "vaderSentiment"
except Exception as _e1:
    try:
        from nltk.sentiment.vader import SentimentIntensityAnalyzer as _SIA
        _an = _SIA()
        _an.polarity_scores("a good test")        # raises if lexicon absent
        _scorer = lambda t: _an.polarity_scores(str(t))["compound"] if t else np.nan
        SENTIMENT_ENGINE = "nltk-vader"
    except Exception as _e2:
        _scorer = _inline_compound
        SENTIMENT_ENGINE = "inline-lexicon"


def score_sentiment(series):
    """Compound sentiment on [-1, 1]; NaN for empty/absent text."""
    s = series.fillna("").astype(str)
    out = s.map(lambda t: _scorer(t) if t.strip() else np.nan)
    return pd.to_numeric(out, errors="coerce")


print(f"SENTIMENT_ENGINE = {SENTIMENT_ENGINE}")
print(score_sentiment(pd.Series([
    "We lost the entire flock to bird flu. Absolutely devastating.",
    "Fresh eggs available today, the girls are healthy and happy!",
    "Routine biosecurity update for the barn.",
    "",
])).round(3).tolist())

<!-- Last changed: Facebook_20260825B -->
### 05A — Parity report

Run this before anyone spends budget. It answers one question: *will this environment
produce numbers that combine with the other two?*

The sentiment fingerprint is the point. `SENTIMENT_ENGINE` degrades silently through
three fallbacks, and a run scored by the inline lexicon is not interchangeable with one
scored by `vaderSentiment` — but every downstream number still computes and every sanity
check still passes. Scoring fixed strings and comparing the scores catches the
divergence in seconds instead of at combine time.

> ⚠️ **This report is safe to send over Slack or email; a row count is not.** Versions,
> engine names, scores of hardcoded strings, and pass/fail booleans are not Meta data.
> Anything counted from an API response is.


In [ ]:
# BLOCK 05A - Parity report (safe to share: no Meta-derived data)
# Last changed: Facebook_20260825E
_parity, _ok = {}, True

# --- environment ------------------------------------------------------------
import platform
import sys as _sys

_parity["python"] = platform.python_version()
_parity["pandas"] = pd.__version__
_parity["numpy"] = np.__version__
_parity["platform"] = platform.system()

try:
    import metacontentlibraryapi as _mcl
    _parity["sdk"] = getattr(_mcl, "__version__", "present (no __version__)")
except Exception as e:
    _parity["sdk"] = f"IMPORT FAILED: {type(e).__name__}"
    _ok = False

_parity["client_usable"] = all(hasattr(client, m) for m in
                               ("get", "post", "has_next_page", "query_next_page"))
_ok &= bool(_parity["client_usable"])

# --- sentiment fingerprint --------------------------------------------------
# Hardcoded strings, so these scores carry no Meta data. Two environments that
# agree here are scoring the corpus the same way; two that disagree are not.
_PROBES = [
    "we lost the entire flock to avian influenza, devastating",
    "beautiful morning at the farm, the hens are happy",
    "HPAI confirmed. Quarantine in effect until further notice.",
    "",
]
_parity["sentiment_engine"] = SENTIMENT_ENGINE
try:
    _scores = score_sentiment(pd.Series(_PROBES, dtype=object))
    _parity["fingerprint"] = " ".join(
        "NaN" if pd.isna(v) else f"{v:+.4f}" for v in _scores)
except Exception as e:
    _parity["fingerprint"] = f"SCORING FAILED: {type(e).__name__}: {e}"
    _ok = False

# --- checkpoint round trip --------------------------------------------------
# Proves the resume path works HERE, before it matters at hour nine of a poll.
try:
    _probe = {"round": "trip", "unicode": "avian influenza — é", "n": 1}
    _ckpt_write("_parity_probe", _probe)
    _back = _ckpt_read("_parity_probe")
    _rt = _back == _probe

    _df_probe = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]})
    _ckpt_write("_parity_frame", _df_probe)
    _rt &= _ckpt_read("_parity_frame").equals(_df_probe)

    for _f in ("_parity_probe.json", "_parity_frame.pkl.gz"):
        _p = os.path.join(CKPT_DIR, _f)
        if os.path.exists(_p):
            os.remove(_p)
    _parity["checkpoint_roundtrip"] = _rt
    _ok &= _rt
except Exception as e:
    _parity["checkpoint_roundtrip"] = f"FAILED: {type(e).__name__}: {e}"
    _ok = False

_parity["ckpt_dir_writable"] = os.access(CKPT_DIR, os.W_OK)
_ok &= bool(_parity["ckpt_dir_writable"])
_parity["shard"] = SHARD
_parity["categories"] = ",".join(PageCategories)
_parity["notebook"] = NOTEBOOK_VERSION

print("=" * 62)
print(f"PARITY REPORT   shard={SHARD}   {'PASS' if _ok else 'FAIL'}")
print("=" * 62)
for _k, _v in _parity.items():
    print(f"  {_k:<22} {_v}")
print("=" * 62)
print("Safe to paste: versions, engine, fixed-string scores, booleans.")
print("NOT safe to paste: any count taken from an API response.")
if _parity["sentiment_engine"] != "vaderSentiment":
    print(f"\nNOTE: sentiment engine is '{_parity['sentiment_engine']}', not "
          f"vaderSentiment.\nThat is fine PROVIDED every shard reports the same "
          f"engine and the same\nfingerprint - compare before anyone starts. It is "
          f"NOT fine to describe the\nresulting scores as VADER scores in the "
          f"methods write-up.")

if not _ok:
    print("\n!! Do not start a production shard until this reads PASS.")
elif IS_SMOKE:
    print("\nEnvironment looks good. Run blocks 06-19 on Zoo to prove the full\n"
          "pipeline, then switch SHARD to your production name.")


<!-- Last changed: Facebook_20260825A -->
## 2 — Pages

Only Pages carry geography, so page discovery has to come first. Country and category
are filtered client-side, not in the request: the API's `admin_countries` is about who
administers the Page, and `categories` is not an exact-primary match.

Discovery is partitioned because it has to be. `facebook/pages/preview` is
synchronous-only, so any single search is capped at 1,000 results in *both* run modes.
Block 07 runs up to three levels:

1. one search per `PageCategories` entry;
2. for any category still returning 1,000 rows, a re-run split across the 16
   `PageDescription` terms;
3. optionally (`PAGE_REFINE_CAPPED`), for any *(category, term)* slice still at 1,000,
   a re-run as `(term & refiner)` AND-groups across `PAGE_REFINERS`.

Every slice is recorded in `PAGE_SLICES`, deduplicated by page id. **Block 07 is meant
to be collapsed** — it prints one line per slice. Block 08 recomputes the funnel and the
truncation report from `PAGE_SLICES` and prints only that.


In [ ]:
# BLOCK 06 - Page search helpers
# Last changed: Facebook_20260825G
# Page discovery is SYNCHRONOUS ONLY - there is no facebook/pages/job endpoint -
# so every individual search inherits the 1,000-result ceiling, in both modes.
SYNC_SEARCH_CAP = 1000

PAGE_FIELDS = ("id,name,page_categories,about,follower_count,"
               "location.country_code,location.state,location.city,location.zip")


def build_ORterms(terms):
    """Concatenate search terms with | as an OR term.

    The API matches token-by-token with no phrase support, so a multi-word term
    must become an AND group. That creates a term-distance problem, fixed
    client-side by the has_post_term regex in block 14.

    Reserved chars (& | - ( ) and space) are not allowed inside a keyword.
    """
    groups = []
    for t in terms:
        tokens = [tok for tok in re.split(r"[^0-9A-Za-z]+", t) if tok]
        if tokens:
            groups.append(tokens[0] if len(tokens) == 1
                          else "(" + " & ".join(tokens) + ")")
    return " | ".join(groups)


def _with_retry(fn, what):
    """Run one page-search API call, retrying transient failures.

    facebook/pages/preview returns HTTP 500 intermittently and a slice that
    gives up contributes ZERO pages - silent, systematic loss rather than a
    visible error. Retried at the CALL level, not the slice level, so a failure
    on page 40 of 100 does not discard the first 39.
    """
    delay = PAGE_SEARCH_RETRY_SLEEP
    for attempt in range(1, PAGE_SEARCH_RETRIES + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == PAGE_SEARCH_RETRIES:
                raise
            print(f"      {what}: {type(e).__name__} on attempt {attempt}/"
                  f"{PAGE_SEARCH_RETRIES}, retrying in {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


def search_pages(categories, q, max_api_pages=None):
    """One synchronous page search, paginated. Returns (rows, hit_cap, error).

    Paging stops at SYNC_SEARCH_CAP because the API will not serve past it; a
    slice that reaches the cap is TRUNCATED, not complete, and says so.

    A call that exhausts its retries returns the rows it had ALREADY paged plus
    an error string, rather than raising. Throwing away 40 pages of results
    because page 41 returned a 500 is a silent, systematic loss - and on this
    endpoint the slices that fail are not random with respect to the research
    question. `error is not None` is the caller's signal that the slice is
    incomplete by ABSENCE rather than by truncation.
    """
    params = {"q": q,
              "admin_countries": PageCountry_Code,   # comma list, not operators
              "fields": PAGE_FIELDS,
              "categories": categories}
    rows = []

    try:
        resp = _with_retry(
            lambda: client.get(path=PAGES_PREVIEW_PATH, params=params),
            "initial search")
    except Exception as e:
        return rows, False, f"{type(e).__name__}: {e}"

    rows = list(resp.json().get("data", []))
    n_api = 1

    while client.has_next_page(resp) and len(rows) < SYNC_SEARCH_CAP:
        if max_api_pages is not None and n_api >= max_api_pages:
            break
        time.sleep(PAGE_SEARCH_SLEEP_S)      # stay under 60 queries/min
        try:
            resp = _with_retry(lambda: client.query_next_page(resp),
                               f"page {n_api + 1}")
        except Exception as e:
            return (rows, len(rows) >= SYNC_SEARCH_CAP,
                    f"{type(e).__name__}: {e} (after {n_api} api page(s))")
        rows.extend(resp.json().get("data", []))
        n_api += 1

    return rows, len(rows) >= SYNC_SEARCH_CAP, None


def slice_query(level, term):
    """Rebuild the exact query a slice used, so an errored slice can be repaired."""
    if level == 1:
        return build_ORterms(PageDescription)
    if level == 2:
        return build_ORterms([term])
    _t, _r = term.split("+", 1)
    return build_ORterms([f"{_t} {_r}"])


print(f"sync search cap : {SYNC_SEARCH_CAP:,} results per search")
print(f"page fields     : {len(PAGE_FIELDS.split(','))} requested")
print(f"OR query        : {build_ORterms(PageDescription)[:110]} ...")

In [ ]:
# BLOCK 07 - Page discovery - RUN (verbose; collapse me, read block 08)
# Last changed: Facebook_20260825G
# Discovery is the slowest block in the notebook - synchronous, rate-limited, and
# hours long at full scope. Restoring it rather than repeating it is the point of
# the checkpoint layer.
#
# 20260825G - the restore path was all-or-nothing, and that hid a real failure. A
# checkpoint written under a NARROWER DiscoveryCategories was reused verbatim, so
# turning DISCOVER_ALL_CATEGORIES on had no effect whatsoever. That is exactly
# what happened on the 2026-08-25 nick run: 18 slices = 2 L1 + 16 L2, shard-only,
# with the widening silently skipped. Scope is now read back off the level-1
# slices and only the GAP is searched.
PAGE_ROWS   = _ckpt_init("PAGE_ROWS", [])    # every row from every slice, pre-dedup
PAGE_SLICES = _ckpt_init("PAGE_SLICES", [])  # one record per search


def _run_slice(cat, term, q, level, parent=None):
    """Run one slice, record it, and return (hit_cap, errored).

    A slice that fails keeps whatever it paged before dying - search_pages
    returns partial rows with an error instead of discarding them. `capped` is
    None on an errored slice, which is how block 08 tells absence from
    truncation.
    """
    rows, capped, err = search_pages(cat, q, MAX_API_PAGES)
    PAGE_ROWS.extend(rows)
    rec = {"category": cat, "term": term, "level": level, "rows": len(rows),
           "capped": None if err else capped, "sub_split": False,
           "narrowed": False}
    if err:
        rec["error"] = err
    PAGE_SLICES.append(rec)
    print(f"  L{level} {cat:<24} {term:<22} {len(rows):>5,} rows"
          + ("  <-- CAPPED" if capped and not err else "")
          + (f"  <-- FAILED {err}" if err else ""))

    # A term slice that dies wholesale gets a SMALLER query, not another
    # identical one. See PAGE_NARROW_FAILED in block 02 for why a repeat failure
    # is treated as deterministic rather than transient.
    if err and PAGE_NARROW_FAILED and level == 2 and not IS_PREVIEW:
        print(f"      -> {cat}/{term} failed; narrowing across "
              f"{len(PAGE_REFINERS)} refiners")
        rec["narrowed"] = True
        for _ref in PAGE_REFINERS:
            _run_slice(cat, f"{term}+{_ref}",
                       build_ORterms([f"{term} {_ref}"]), level=3)

    return capped, bool(err)


def _discover_categories(cats):
    """L1 sweep over `cats`, sub-splitting any category that caps or fails.

    Factored out so incremental widening and a cold start run the SAME code. A
    page universe assembled across two runs must not have been built by two
    different code paths.
    """
    _q_pages = build_ORterms(PageDescription)

    for _cat in cats:
        _capped_cat, _err_cat = _run_slice(_cat, "<all terms>", _q_pages, level=1)
        _parent = len(PAGE_SLICES) - 1
        if not ((_capped_cat or _err_cat) and not IS_PREVIEW):
            continue

        # Only a category that is still truncated - or that died wholesale - is
        # worth re-querying per term. The parent slice is EXPECTED to be capped
        # here, which is what triggered the sub-split, so it is excluded from
        # the truncation report in block 08.
        print(f"      -> {_cat} {'failed' if _err_cat else 'capped'}; L2 split "
              f"across {len(PageDescription)} terms")
        PAGE_SLICES[_parent]["sub_split"] = True

        for _term in PageDescription:
            _capped_term, _ = _run_slice(_cat, _term, build_ORterms([_term]),
                                         level=2)
            _parent2 = len(PAGE_SLICES) - 1
            if not (_capped_term and PAGE_REFINE_CAPPED):
                continue

            # L3. On 2026-08-24, 13 L2 slices were still capped - Farm and
            # Agricultural Service across livestock/farm/farming/ranch/
            # agriculture/agricultural. AND-refining narrows them, but a page
            # carrying the term and none of the refiners stays invisible.
            print(f"          -> {_cat}/{_term} capped; L3 across "
                  f"{len(PAGE_REFINERS)} refiners")
            PAGE_SLICES[_parent2]["sub_split"] = True
            for _ref in PAGE_REFINERS:
                _run_slice(_cat, f"{_term}+{_ref}",
                           build_ORterms([f"{_term} {_ref}"]), level=3)


# What this checkpoint has ACTUALLY searched, read back off its level-1 slices
# rather than taken from config - config is the thing someone just changed.
SEARCHED_CATEGORIES = sorted({c.strip()
                              for r in PAGE_SLICES if r.get("level") == 1
                              for c in str(r.get("category", "")).split(",")
                              if c.strip()})
_missing = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]

# A slice that has already been narrowed is not retried: its replacement L3
# slices are sitting in PAGE_SLICES, and re-running it would duplicate them on
# every pass. RESTORE_FROM_CHECKPOINT = False forces a clean sweep instead.
_errored = [r for r in PAGE_SLICES
            if r.get("capped") is None and not r.get("narrowed")]

if PAGE_ROWS and (_missing or (_errored and RETRY_FAILED_SLICES)):
    print(f"{len(PAGE_ROWS):,} page rows present across "
          f"{len(SEARCHED_CATEGORIES)} searched category(ies).\n")

    if _errored and RETRY_FAILED_SLICES:
        # Repair in place. A slice that errored contributed nothing, and redoing
        # the whole of discovery to recover four slices would cost hours.
        print(f"  {len(_errored)} slice(s) errored - repairing those only.\n")
        for _rec in list(_errored):
            PAGE_SLICES.remove(_rec)
            _run_slice(_rec["category"], _rec["term"],
                       slice_query(_rec.get("level", 1), _rec["term"]),
                       level=_rec.get("level", 1))
        checkpoint("pages-repaired")

    if _missing:
        # Incremental widening. Everything already discovered is kept and only
        # the categories this checkpoint never saw are searched, so turning
        # DISCOVER_ALL_CATEGORIES on costs four category sweeps rather than a
        # from-scratch re-run of all of discovery.
        print(f"\n  scope gap: {len(_missing)} of {len(DiscoveryCategories)} "
              f"discovery category(ies) never searched -> {_missing}\n")
        _discover_categories(_missing)
        SEARCHED_CATEGORIES = sorted(set(SEARCHED_CATEGORIES) | set(_missing))
        checkpoint("pages-widened")

elif PAGE_ROWS:
    print(f"{len(PAGE_ROWS):,} page rows already present, all "
          f"{len(DiscoveryCategories)} discovery category(ies) searched, "
          f"nothing left\nto repair - skipping discovery. Set "
          f"RESTORE_FROM_CHECKPOINT = False to force a fresh search.\n")

elif PARTITION_PAGE_SEARCH:
    print(f"partitioned page discovery: {len(DiscoveryCategories)} category "
          f"slices"
          + ("  (preview: no sub-splitting)" if IS_PREVIEW else "")
          + "\n")
    _discover_categories(DiscoveryCategories)
    SEARCHED_CATEGORIES = sorted(DiscoveryCategories)

else:
    print("unpartitioned page discovery - capped at 1,000 rows\n")
    _run_slice(",".join(DiscoveryCategories), "<all terms>",
               build_ORterms(PageDescription), level=1)
    SEARCHED_CATEGORIES = sorted(DiscoveryCategories)

checkpoint("pages-discovered")

In [ ]:
# BLOCK 08 - Page discovery - SUMMARY
# Last changed: Facebook_20260825G
# Everything block 07 printed line-by-line, recomputed from PAGE_SLICES so the
# verbose block above can stay collapsed.
page_slices = pd.DataFrame(PAGE_SLICES)
for _col, _fill in (("error", None), ("narrowed", False), ("sub_split", False)):
    if _col not in page_slices.columns:
        page_slices[_col] = _fill
if len(page_slices):
    page_slices["narrowed"] = page_slices["narrowed"].fillna(False).astype(bool)
    page_slices["sub_split"] = page_slices["sub_split"].fillna(False).astype(bool)

NormalizedRows = (pd.json_normalize(PAGE_ROWS).drop_duplicates("id")
                  if PAGE_ROWS else pd.DataFrame(columns=["id"]))

print(f"{len(PAGE_ROWS):,} rows across {len(page_slices)} slice(s) "
      f"-> {len(NormalizedRows):,} unique pages")

# Scope comes FIRST, because it silently rescales everything under it. The
# categories filter is not an exact-primary match and every slice caps at 1,000,
# so a page whose PRIMARY category is Farm can be reachable only through some
# other category's slice: searching fewer categories finds fewer of your OWN
# pages. A count from a narrow sweep is not comparable to one from a wide sweep.
print(f"\ndiscovery scope : {len(SEARCHED_CATEGORIES)} of "
      f"{len(DiscoveryCategories)} category(ies) searched")
print(f"                  {SEARCHED_CATEGORIES}")
_gap = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]
if _gap:
    print(f"!! {len(_gap)} never searched: {_gap}")
    print("!! Every page and post count below is a FLOOR, the block 12A estimate "
          "included.\n   Re-run block 07 to close the gap before planning "
          "collection.")

if len(page_slices):
    _by_level = (page_slices.groupby("level")
                 .agg(slices=("rows", "size"), rows=("rows", "sum"),
                      capped=("capped", lambda s: int((s == True).sum())),  # noqa: E712
                      failed=("capped", lambda s: int(s.isna().sum()))))
    print("\nby split level:")
    print(_by_level.to_string())

# A sub-split parent is superseded by its child slices; only unresolved
# truncation counts as truncation.
_capped_slices = (page_slices[(page_slices["capped"] == True)      # noqa: E712
                              & (~page_slices["sub_split"])]
                  if len(page_slices) else pd.DataFrame())
_failed_slices = (page_slices[page_slices["capped"].isna()]
                  if len(page_slices) else pd.DataFrame())

if len(_capped_slices):
    print(f"\n!! {len(_capped_slices)} slice(s) still at the {SYNC_SEARCH_CAP:,} cap - "
          f"truncated by an unknown amount:")
    print(_capped_slices[["level", "category", "term", "rows"]].to_string(index=False))
    print("!! The page universe is a SAMPLE in these slices, not a census. "
          "Record this in the methods write-up.")
    if not PAGE_REFINE_CAPPED:
        print("!! PAGE_REFINE_CAPPED is off - set it True to split these further, "
              "at the cost of a larger and more expensive surface list.")
else:
    print("\nno unresolved slice hit the cap")

if len(_failed_slices):
    _partial = int(_failed_slices["rows"].sum())
    _narrowed = int(_failed_slices["narrowed"].sum())
    print(f"\n!! {len(_failed_slices)} slice(s) errored ({_partial:,} row(s) "
          f"paged before failure were KEPT):")
    print(_failed_slices[["level", "category", "term", "rows", "narrowed",
                          "error"]].to_string(index=False))
    print("!! This is not truncation, it is absence, and it is not random: the "
          "terms that\n   fail are as likely to be the poultry-specific ones as "
          "any other.")
    if _narrowed:
        print(f"!! {_narrowed} of them were re-queried as narrowed "
              f"'(term & refiner)' slices, so they\n   are partly covered - but "
              f"a page carrying the term and none of the refiners\n   is still "
              f"invisible. Say so in the methods write-up.")
    if len(_failed_slices) > _narrowed:
        print("!! The rest contributed only their partial rows. Re-run block 07 "
              "with\n   RETRY_FAILED_SLICES = True to repair just those - "
              "everything already\n   discovered is kept.")

In [ ]:
# BLOCK 09 - Page filter -> SourcePages
# Last changed: Facebook_20260825G
# Country and PRIMARY-category filtering happen HERE, not in the request.
us = NormalizedRows[NormalizedRows.get("location.country_code",
                                       pd.Series(dtype=object)).fillna("") == "US"]

want = {c.lower() for c in PageCategories}


def primary_category(cell):
    """First element of page_categories, or None."""
    items = cell if isinstance(cell, (list, tuple)) else []
    return str(items[0]) if items else None


us = us.assign(page_category=us["page_categories"].apply(primary_category)
               if "page_categories" in us.columns else None)
CleanUSPages = us[us["page_category"].fillna("").str.lower().isin(want)].copy()

FarmPages = CleanUSPages[CleanUSPages["page_category"].str.lower() == "farm"]

SourcePages = FarmPages if RESTRICT_TO_FARM_ONLY else CleanUSPages

# Preview mode truncates the surface list so the post pull stays a single batch.
if MAX_SURFACES is not None and len(SourcePages) > MAX_SURFACES:
    print(f"{RUN_MODE}: capping surfaces {len(SourcePages):,} -> {MAX_SURFACES}")
    SourcePages = SourcePages.head(MAX_SURFACES)

print(f"{len(NormalizedRows):,} pages -> {len(us):,} US-located "
      f"-> {len(CleanUSPages):,} with a wanted PRIMARY category "
      f"-> {len(FarmPages):,} Farms")
print(f"sourcing posts from {len(SourcePages):,} pages "
      f"({'Farm only' if RESTRICT_TO_FARM_ONLY else 'all wanted categories'})\n")
if len(CleanUSPages):
    print(CleanUSPages["page_category"].value_counts().to_string())

# Drift check against the 2026-08-24 discovery run. The comparison is
# ASYMMETRIC on purpose, because the two directions mean different things.
#
#   SHORTFALL is a problem. It means a truncated or errored page universe, and
#   the post counts built on it are floors that the combined dataset inherits
#   silently.
#
#   SURPLUS is the expected outcome, not an anomaly. The baseline was measured
#   over 8 categories with four Farm slices erroring to ZERO, so it is itself a
#   floor. Wide discovery plus slice repair is supposed to beat it: the
#   2026-08-25 nick run reached 4,246 Farm against a 3,837 baseline (+10.7%)
#   with only two categories searched, and flagging that as drift is a false
#   alarm that trains you to ignore the flag.
print("\nvs 2026-08-24 baseline (itself a FLOOR: 8 categories, 4 Farm slices "
      "errored to zero)")
print(f"   this run searched {len(SEARCHED_CATEGORIES)} of "
      f"{len(DiscoveryCategories)} discovery category(ies)")
_obs = (CleanUSPages["page_category"].value_counts()
        if len(CleanUSPages) else pd.Series(dtype=int))
for _c in PageCategories:
    _exp = PageCategoryPages.get(_c, 0)
    _got = int(_obs.get(_c, 0))
    _pct = (_got / _exp - 1) * 100 if _exp else 0.0
    if _exp and _pct < -10:
        _flag = "  <-- SHORT, investigate"
    elif _exp and _pct > 10:
        _flag = "  (above floor - expected)"
    else:
        _flag = ""
    print(f"  {_c:<26} floor {_exp:>6,}   now {_got:>6,}   {_pct:+6.1f}%{_flag}")

# A page belongs to exactly one shard because this filter is on PRIMARY category.
_stray = set(CleanUSPages["page_category"]) - set(PageCategories)
assert not _stray, f"pages outside this shard's categories leaked through: {_stray}"

<!-- Last changed: Facebook_20260825D -->
## 3 — Posts

`SourcePosts` is kept **whole**. The precise `PostText` concepts are applied later as a
flag, never as a row filter — that is the point of this notebook.

Block 10 only *defines* the fetch machinery. Which path executes depends on `RUN_MODE`:

* **preview** → `client.get("facebook/posts/preview")` per surface batch, paginated,
  capped at 1,000 results per batch.
* **full** → `client.post("facebook/posts/job")` per **(surface batch × time window)**,
  then polling. Submissions are spaced by `JOB_SUBMIT_SLEEP_S` because async search is
  limited to one query per minute.

### The job plan

A *job key* is `b{batch:03d}|{window_start}` — a string, deliberately, so it survives a
JSON round-trip into the checkpoint without hand-coercion. `_job_plan()` is the single
source of truth for what work exists; estimation, submission, and polling all walk it.

### The budget ledger

500,000 records per rolling 7 days. `BUDGET_LEDGER` records what each submission was
estimated to cost and when, `_budget_spent_7d()` sums the live window, and
`submit_post_jobs` stops rather than overrunning it. The ledger is checkpointed, so
next week's session resumes the staged collection instead of restarting it.


In [ ]:
# BLOCK 10 - Post fetch machinery
# Last changed: Facebook_20260825A
SURFACE_BATCH = 250          # documented max IDs per request
CAP = SYNC_SEARCH_CAP        # sync ceiling; async jobs use JOB_RESULT_CAP

POST_FIELDS = ("id,creation_time,text,lang,surface.id,post_owner_id,"
               "post_owner_type,statistics")

# Module-level so state survives a crash mid-run; _ckpt_init extends that to
# surviving the kernel itself.
POST_ROWS      = _ckpt_init("POST_ROWS", [])       # raw post payloads
POST_BATCH     = _ckpt_init("POST_BATCH", [])      # per-job diagnostics
POST_JOBS      = _ckpt_init("POST_JOBS", {})       # job_key -> job metadata
POST_ESTIMATES = _ckpt_init("POST_ESTIMATES", {})  # job_key -> estimated records
BUDGET_LEDGER  = _ckpt_init("BUDGET_LEDGER", [])   # submissions, for the rolling window


def _epoch(d, end_of_day=False):
    """The async job endpoint's documented samples pass Unix seconds, not ISO dates."""
    ts = pd.Timestamp(d, tz="UTC")
    if end_of_day:
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return int(ts.timestamp())


def _base_params(chunk, q, fields):
    p = {"lang": "en",
         "search_scope": "post_text_only",
         "surface_ids": chunk,
         "fields": fields}
    if q:                        # omitted entirely in "all_posts" mode
        p["q"] = q
    return p


def _batches(page_ids):
    page_ids = list(page_ids)
    for i in range(0, len(page_ids), SURFACE_BATCH):
        yield i // SURFACE_BATCH, page_ids[i:i + SURFACE_BATCH]


def _job_plan(page_ids):
    """Every (surface batch x time window) unit of work, in submission order.

    Windowing is what keeps a job under JOB_RESULT_CAP: one job per batch over
    the whole window averaged ~115,000 posts on the 2026-08-24 estimate, against
    a 100,000 ceiling.
    """
    plan = []
    for b, chunk in _batches(page_ids):
        for since, until in JOB_WINDOWS:
            plan.append({"job_key": f"b{b:03d}|{since}", "batch": b,
                         "chunk": chunk, "since": since, "until": until})
    return plan


# ---------------------------------------------------------------------------
# BUDGET
# ---------------------------------------------------------------------------
def _budget_spent_7d(now=None):
    """Estimated records submitted inside the live rolling window."""
    now = time.time() if now is None else now
    cut = now - 7 * 86400
    return sum(e.get("estimated", 0) for e in BUDGET_LEDGER
               if float(e.get("at", 0)) >= cut)


def _budget_remaining():
    return max(0, RECORD_BUDGET - _budget_spent_7d())


def _budget_note(e):
    """Human-readable age of a ledger entry."""
    hrs = (time.time() - float(e.get("at", 0))) / 3600
    return f"{hrs:5.1f} h ago  {e.get('estimated', 0):>8,}  {e.get('job_key', '')}"


# ---------------------------------------------------------------------------
# SYNCHRONOUS path (RUN_MODE == "preview")
# ---------------------------------------------------------------------------
def fetch_posts_sync(page_ids, q, since, until, fields=POST_FIELDS):
    """Paginated sync search per surface batch. Resumable - rerun skips done work."""
    done = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}

    for b, chunk in _batches(page_ids):
        key = f"b{b:03d}|{since}"
        if key in done:
            print(f"  {key}  skip (already done)")
            continue
        try:
            params = _base_params(chunk, q, fields)
            params["since"], params["until"] = since, until

            resp = client.get(path=POSTS_PREVIEW_PATH, params=params)
            recs = list(resp.json().get("data", []))
            while client.has_next_page(resp):
                resp = client.query_next_page(resp)
                recs.extend(resp.json().get("data", []))

            POST_ROWS.extend(recs)
            POST_BATCH.append({"job_key": key, "batch": b, "surfaces": len(chunk),
                               "returned": len(recs), "capped": len(recs) >= CAP})
            print(f"  {key}  {len(recs):>5,} posts"
                  f"{'  <-- CAPPED' if len(recs) >= CAP else ''}"
                  f"   running {len(POST_ROWS):,}")

        except Exception as e:
            POST_BATCH.append({"job_key": key, "batch": b, "surfaces": len(chunk),
                               "returned": None, "capped": None,
                               "error": f"{type(e).__name__}: {e}"})
            print(f"  {key}  FAILED  {type(e).__name__}: {e}")


# ---------------------------------------------------------------------------
# ASYNCHRONOUS path (RUN_MODE == "full")
# ---------------------------------------------------------------------------
def _job_id(payload):
    """Pull the job id out of a /job POST response.

    The docs display the response payload but never name the key, so try the
    plausible shapes and fail loudly with the raw payload rather than guessing.
    """
    if hasattr(payload, "json"):
        payload = payload.json()
    if isinstance(payload, dict):
        for k in ("id", "job_id", "async_job_id", "request_id"):
            if payload.get(k):
                return str(payload[k])
        for k in ("data", "job", "result"):
            inner = payload.get(k)
            if isinstance(inner, list) and inner:
                inner = inner[0]
            if isinstance(inner, dict):
                for kk in ("id", "job_id", "async_job_id"):
                    if inner.get(kk):
                        return str(inner[kk])
    raise KeyError(f"no job id found in submit response: {payload!r}")


def _status_str(st):
    """Normalize get_status() output - dict, response, or bare string."""
    if hasattr(st, "json"):
        try:
            st = st.json()
        except Exception:
            pass
    if isinstance(st, dict):
        for k in ("status", "state", "job_status"):
            if st.get(k):
                return str(st[k]).upper()
    return str(st).upper()


def _job_records(data):
    """Normalize get_data() output to a list of post dicts."""
    if hasattr(data, "json"):
        try:
            data = data.json()
        except Exception:
            pass
    if isinstance(data, dict):
        for k in ("data", "records", "results"):
            if isinstance(data.get(k), list):
                return data[k]
        return []
    return list(data) if isinstance(data, (list, tuple)) else []


def _estimate_count(est):
    """Pull a numeric count out of an /estimate payload of unknown shape."""
    if hasattr(est, "json"):
        try:
            est = est.json()
        except Exception:
            return None
    candidates = [est]
    if isinstance(est, dict):
        for k in ("data", "result", "estimate"):
            v = est.get(k)
            if isinstance(v, list) and v:
                v = v[0]
            if isinstance(v, dict):
                candidates.append(v)
    for c in candidates:
        if isinstance(c, dict):
            for k in ("estimated_count", "estimated_results", "count", "estimate",
                      "total_count", "upper_bound"):
                v = c.get(k)
                if isinstance(v, bool):
                    continue
                if isinstance(v, (int, float)):
                    return int(v)
    return None


def estimate_post_jobs(plan, q, fields=POST_FIELDS, verbose=False):
    """Estimate every planned job. Spends no RESULT budget, but does spend calls.

    Returns (total, n_unparsed). A non-zero n_unparsed means the payload shape is
    not one _estimate_count recognizes, so the total is a FLOOR.
    """
    total, unparsed = 0, 0
    for job in plan:
        key = job["job_key"]
        if key in POST_ESTIMATES:
            total += POST_ESTIMATES[key]
            continue
        try:
            params = _base_params(job["chunk"], q, fields)
            params["since"], params["until"] = job["since"], job["until"]
            est = client.get(path=POSTS_ESTIMATE_PATH, params=params).json()
            n = _estimate_count(est)
            if n is None:
                unparsed += 1
                print(f"  {key}  estimate NOT PARSED: {est}")
            else:
                POST_ESTIMATES[key] = n
                total += n
                if verbose:
                    print(f"  {key}  {n:>8,}")
        except Exception as e:
            unparsed += 1
            print(f"  {key}  estimate FAILED  {type(e).__name__}: {e}")
        time.sleep(PAGE_SEARCH_SLEEP_S)      # estimate rides the sync rate limit
    return total, unparsed


def submit_post_jobs(plan, q, fields=POST_FIELDS):
    """Submit planned jobs, budget-gated. Already-submitted keys are skipped.

    Stops - rather than overrunning - when the next job's estimate exceeds what
    the rolling 7-day window has left. The ledger is checkpointed, so a later
    session resumes the staged collection.
    """
    submitted = 0
    for job in plan:
        key = job["job_key"]
        if key in POST_JOBS:
            continue
        if submitted >= MAX_JOBS_PER_RUN:
            print(f"\n  MAX_JOBS_PER_RUN ({MAX_JOBS_PER_RUN}) reached - stopping. "
                  f"Re-run this block to continue.")
            return submitted

        est = POST_ESTIMATES.get(key)
        remaining = _budget_remaining()
        if est is not None and est > remaining:
            print(f"\n  BUDGET STOP at {key}: next job ~{est:,} records, "
                  f"{remaining:,} left in the rolling 7-day window.\n"
                  f"  {len(plan) - submitted:,} job(s) still unsubmitted. Re-run "
                  f"when the window frees up; the ledger is checkpointed.")
            return submitted

        try:
            params = _base_params(job["chunk"], q, fields)
            params["since"] = _epoch(job["since"])
            params["until"] = _epoch(job["until"], end_of_day=True)
            params["mode"] = JOB_MODE
            params["name"] = f"{JOB_NAME} [{key}]"
            params["description"] = JOB_DESCRIPTION

            resp = client.post(path=POSTS_JOB_PATH, params=params)
            if not POST_JOBS:                 # show the real payload shape once
                print(f"  submit response payload: {resp.json()}")
            POST_JOBS[key] = {"job_id": _job_id(resp), "batch": job["batch"],
                              "since": job["since"], "until": job["until"],
                              "surfaces": len(job["chunk"]),
                              "status": "IN_PROGRESS", "n": None}
            BUDGET_LEDGER.append({"at": time.time(), "job_key": key,
                                  "estimated": int(est or 0)})
            submitted += 1
            _note = f"   (~{est:,} records)" if est else "   (no estimate)"
            print(f"  {key}  submitted job {POST_JOBS[key]['job_id']}{_note}")
        except Exception as e:
            POST_BATCH.append({"job_key": key, "batch": job["batch"],
                               "surfaces": len(job["chunk"]),
                               "returned": None, "capped": None,
                               "error": f"submit: {type(e).__name__}: {e}"})
            print(f"  {key}  SUBMIT FAILED  {type(e).__name__}: {e}")

        time.sleep(JOB_SUBMIT_SLEEP_S)        # async limit is 1 query/minute
    return submitted


def collect_post_jobs(budget_min=None):
    """Poll submitted jobs, harvesting COMPLETE ones. True when all resolved.

    Bounded by a wall-clock budget because jobs can take days. Re-running is
    safe: harvested keys are recorded in POST_BATCH and skipped.
    """
    budget_min = JOB_POLL_BUDGET_MIN if budget_min is None else budget_min
    deadline = time.time() + budget_min * 60
    harvested = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}

    while True:
        pending = [k for k in POST_JOBS if k not in harvested]
        if not pending:
            return True

        for key in list(pending):
            meta = POST_JOBS[key]
            try:
                job = client.get_async_job(job_id=meta["job_id"])
                status = _status_str(job.get_status())
                meta["status"] = status

                if status.startswith("COMPLETE"):
                    recs = _job_records(job.get_data())
                    POST_ROWS.extend(recs)
                    meta["n"] = len(recs)
                    POST_BATCH.append({"job_key": key, "batch": meta["batch"],
                                       "surfaces": meta["surfaces"],
                                       "returned": len(recs),
                                       "capped": len(recs) >= JOB_RESULT_CAP,
                                       "job_id": meta["job_id"]})
                    harvested.add(key)
                    print(f"  {key}  COMPLETE  {len(recs):>6,} posts"
                          f"{'  <-- CAPPED' if len(recs) >= JOB_RESULT_CAP else ''}"
                          f"   running {len(POST_ROWS):,}")

                elif status.startswith("FAIL") or status.startswith("ERROR"):
                    POST_BATCH.append({"job_key": key, "batch": meta["batch"],
                                       "surfaces": meta["surfaces"],
                                       "returned": None, "capped": None,
                                       "job_id": meta["job_id"],
                                       "error": f"job status {status}"})
                    harvested.add(key)
                    print(f"  {key}  FAILED  job {meta['job_id']} -> {status}")

            except Exception as e:
                print(f"  {key}  poll error  {type(e).__name__}: {e}")

        pending = [k for k in POST_JOBS if k not in harvested]
        if not pending:
            return True
        if time.time() >= deadline:
            print(f"\n  polling budget ({budget_min} min) exhausted; "
                  f"{len(pending)} job(s) still running:")
            for key in pending[:20]:
                print(f"    {key}  {POST_JOBS[key]['job_id']}  "
                      f"{POST_JOBS[key]['status']}")
            if len(pending) > 20:
                print(f"    ... and {len(pending) - 20} more")
            print("  re-run - POST_JOBS is kept, nothing resubmits.")
            return False

        print(f"  {len(pending)} job(s) still running; "
              f"sleeping {JOB_POLL_INTERVAL_S}s")
        time.sleep(JOB_POLL_INTERVAL_S)


print(f"surface batch   : {SURFACE_BATCH}")
print(f"job windows     : {len(JOB_WINDOWS)}")
print(f"restored        : {len(POST_JOBS)} jobs, {len(POST_ROWS):,} rows, "
      f"{len(BUDGET_LEDGER)} ledger entries")


In [ ]:
# BLOCK 11 - Pre-flight - checkpoint, budget, and job status
# Last changed: Facebook_20260825A
print(f"checkpoint dir : {CKPT_DIR}")
for _f in sorted(os.listdir(CKPT_DIR)):
    print(f"  {_f:<28} {os.path.getsize(os.path.join(CKPT_DIR, _f)) / 1e6:>8.2f} MB")

_harvested = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}
print(f"\nPAGE_ROWS {len(PAGE_ROWS):,}   POST_ROWS {len(POST_ROWS):,}   "
      f"POST_JOBS {len(POST_JOBS)}   harvested {len(_harvested)}")

print(f"\nrolling 7-day budget: {_budget_spent_7d():,} spent / {RECORD_BUDGET:,} "
      f"-> {_budget_remaining():,} available")
for _e in sorted(BUDGET_LEDGER, key=lambda e: -float(e.get("at", 0)))[:10]:
    print(f"  {_budget_note(_e)}")

if POST_JOBS and not IS_PREVIEW:
    print(f"\nlive status for {len(POST_JOBS)} job(s) "
          f"(one status call each, no result budget spent):")
    _tally = {}
    for _k, _m in sorted(POST_JOBS.items()):
        try:
            _st = _status_str(client.get_async_job(job_id=_m["job_id"]).get_status())
        except Exception as e:
            _st = f"UNREACHABLE ({type(e).__name__})"
        _m["status"] = _st
        _tally[_st] = _tally.get(_st, 0) + 1
    for _st, _n in sorted(_tally.items()):
        print(f"  {_st:<24} {_n:>4}")
    _ready = [k for k, m in POST_JOBS.items()
              if m["status"].startswith("COMPLETE") and k not in _harvested]
    print(f"  COMPLETE but not yet harvested: {len(_ready)} "
          f"(free to fetch - the records are already paid for)")
else:
    print("\nno submitted jobs on record")


In [ ]:
# BLOCK 12 - Post plan and estimate - RUN (verbose; collapse me, read block 12A)
# Last changed: Facebook_20260825F
_q = None if POST_QUERY_MODE == "all_posts" else build_ORterms(PostSingleWordText)
if _q is None:
    print("POST_QUERY_MODE='all_posts': no q sent, so n_posts is a true denominator.\n"
          "If every job FAILS with a 400, the endpoint requires a query - set\n"
          "POST_QUERY_MODE='broad_terms' and rerun.\n")
else:
    print("POST_QUERY_MODE='broad_terms': n_posts counts broad-term posts only.\n")

ALL_JOBS_DONE = True     # meaningful in full mode only
SUBMIT_GATED  = False    # set by block 12A
PLAN, PLAN_TODO = [], []
_est, _unparsed = 0, 0

if IS_PREVIEW:
    print(f"SYNC pull from {POSTS_PREVIEW_PATH}  ({RUN_START} .. {RUN_END})")
    fetch_posts_sync(SourcePages["id"], q=_q, since=RUN_START, until=RUN_END)
else:
    PLAN = _job_plan(SourcePages["id"])
    PLAN_TODO = [j for j in PLAN if j["job_key"] not in POST_JOBS]

    if RUN_ESTIMATE and PLAN_TODO:
        print(f"ESTIMATE via {POSTS_ESTIMATE_PATH} for {len(PLAN_TODO):,} job(s) "
              f"- spends no result budget. One call each, ~"
              f"{len(PLAN_TODO) * PAGE_SEARCH_SLEEP_S / 60:.0f} min.\n")
        _est, _unparsed = estimate_post_jobs(PLAN_TODO, q=_q)
        checkpoint("estimated")
    elif not PLAN_TODO:
        print("every planned job is already submitted - nothing to estimate.")
    else:
        print("RUN_ESTIMATE is off - no estimate, and no gate. Block 12B will "
              "submit\nblind if CONFIRM_SUBMIT_JOBS is set.")


In [ ]:
# BLOCK 12A - Plan, estimate, and budget SUMMARY
# Last changed: Facebook_20260825G
if IS_PREVIEW:
    print("RUN_MODE='preview' - synchronous pull, no plan and no budget gate.")
else:
    _over = [k for k, v in POST_ESTIMATES.items() if v >= JOB_RESULT_CAP]
    _n_batches = math.ceil(len(SourcePages) / SURFACE_BATCH)

    print(f"plan               : {len(PLAN):,} job(s) = {_n_batches} batch(es) "
          f"x {len(JOB_WINDOWS)} window(s)")
    print(f"already submitted  : {len(PLAN) - len(PLAN_TODO):,}")
    print(f"still to submit    : {len(PLAN_TODO):,}")

    if POST_ESTIMATES:
        print(f"\nestimated remaining: {_est:,} posts across {len(PLAN_TODO):,} "
              f"unsubmitted job(s)")
        print(f"budget             : {RECORD_BUDGET:,} records / rolling 7 days "
              f"-> {_est / RECORD_BUDGET:.1f} budget-week(s)")
        print(f"already spent (7d) : {_budget_spent_7d():,}  "
              f"-> {_budget_remaining():,} available now")
        print(f"jobs over the cap  : {len(_over)} of {len(POST_ESTIMATES)} "
              f"at >= {JOB_RESULT_CAP:,}")
        if _over:
            print(f"  !! those jobs WILL truncate. Narrow JOB_WINDOW "
                  f"('{JOB_WINDOW}' -> 'quarter' -> 'month') and re-plan.")
        if _unparsed:
            print(f"  !! {_unparsed} estimate(s) unparsed - the total is a FLOOR.")

        # Wall-clock, which MAX_JOBS_PER_RUN does not tell you. Submission is
        # bounded by the rolling RECORD budget, not by how many jobs you are
        # willing to fire: at ~23,000 records a job, 500,000 per 7 days admits
        # about 21 of them a week however long you sit here. That is the number
        # that decides whether this shard is a 4-week or a 12-week job.
        _avg = _est / max(len(PLAN_TODO), 1)
        _per_week = int(RECORD_BUDGET // _avg) if _avg else 0
        _weeks = math.ceil(len(PLAN_TODO) / _per_week) if _per_week else 0
        print(f"\nmean job size      : {_avg:,.0f} records")
        print(f"budget admits      : ~{_per_week} job(s) per rolling 7 days "
              f"-> ~{_weeks} week(s) to submit all {len(PLAN_TODO):,}")
        print(f"this session       : at most "
              f"{min(MAX_JOBS_PER_RUN, len(PLAN_TODO)):,} job(s) "
              f"(MAX_JOBS_PER_RUN={MAX_JOBS_PER_RUN}); the budget stops it "
              f"sooner\n                     if the rolling window is already "
              f"spent.")
        print("  !! UNRESOLVED: whether the 500,000/7d budget is scoped per "
              "RESEARCHER or per\n     PROJECT. Per-researcher, the three "
              "shards run concurrently and the week\n     count above is the "
              "real one. Per-project they share it, and the true\n     figure "
              "is roughly three times that. Nobody has checked.")

    print(f"\nscope: {len(SourcePages):,} surfaces from a page universe whose "
          f"slices cap at\n{SYNC_SEARCH_CAP:,}. This is NOT an estimate of all US "
          f"farm-page posts.")

    # The estimate scales with the surface list, so an incomplete page universe
    # produces a total that is low for a reason having nothing to do with how
    # many posts exist. Say which kind of number this is before anyone plans a
    # collection schedule around it.
    _scope_gap = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]
    if _scope_gap:
        print(f"!! discovery is INCOMPLETE - {len(_scope_gap)} category(ies) "
              f"unsearched: {_scope_gap}.\n   Every total above is a floor and "
              f"will grow. Close the gap in block 07\n   before deciding scope.")

    # --- the gate -----------------------------------------------------------
    if RUN_ESTIMATE and PLAN_TODO and not CONFIRM_SUBMIT_JOBS:
        SUBMIT_GATED = True
        ALL_JOBS_DONE = False
        print("\n" + "=" * 62)
        print("STOPPED before submitting - no result budget has been spent.")
        print("Read the numbers above. To proceed: set CONFIRM_SUBMIT_JOBS = True")
        print("in block 02, re-run blocks 02-03, then run block 12B.")
        print("=" * 62)
        print("\nBlocks 13-21 below will run on an EMPTY frame. That is expected;")
        print("nothing is broken and nothing was submitted.")
    elif not PLAN_TODO:
        print("\nnothing left to submit - go to block 12B to keep polling.")
    else:
        print("\nCONFIRM_SUBMIT_JOBS is set - block 12B will submit.")

In [ ]:
# BLOCK 12B - Submit and poll - RUN (verbose; collapse me, read block 13)
# Last changed: Facebook_20260825F
if IS_PREVIEW:
    print("preview mode - block 12 already fetched synchronously, nothing to submit.")
elif SUBMIT_GATED:
    print("the estimate gate is closed - nothing submitted. See block 12A.")
else:
    print(f"ASYNC submit to {POSTS_JOB_PATH}"
          f"  (<= {MAX_JOBS_PER_RUN} job(s) x {JOB_SUBMIT_SLEEP_S}s spacing "
          f"~ {MAX_JOBS_PER_RUN * JOB_SUBMIT_SLEEP_S / 3600:.1f} h)\n")
    _n_sub = submit_post_jobs(PLAN, q=_q)

    # Job ids before anything else: they are the only handle on results Meta
    # already holds, and losing them means re-spending the query budget.
    checkpoint("post-submit")

    print(f"\npolling: up to {POLL_ROUNDS} round(s) x {JOB_POLL_BUDGET_MIN} min "
          f"(~{POLL_ROUNDS * JOB_POLL_BUDGET_MIN / 60:.1f} h), "
          f"checkpointing after each")
    for _round in range(POLL_ROUNDS):
        print(f"\n  -- poll round {_round + 1}/{POLL_ROUNDS} --")
        ALL_JOBS_DONE = collect_post_jobs()
        checkpoint(f"poll-{_round + 1}")
        if ALL_JOBS_DONE:
            break

checkpoint("posts-fetched")


In [ ]:
# BLOCK 13 - Post pull - SUMMARY
# Last changed: Facebook_20260825D
# Everything block 12 printed job-by-job, recomputed so it can stay collapsed.
#
# NAMING (renamed in 20260825D). This frame was called FarmPosts through
# 20260825C, which was a leftover from H5N1_20260819 when PageCategories was
# ["Farm"]. It has never been Farm-only since: it holds every post from every
# page in SourcePages, which is CleanUSPages - all of this shard's categories -
# unless RESTRICT_TO_FARM_ONLY is set. SourcePages -> SourcePosts now reads as
# the lineage it actually is. FarmPages keeps its name; that one really is
# Farm-only.
batch_report = pd.DataFrame(POST_BATCH)

# json_normalize([]) returns a frame with NO columns, so drop_duplicates("id")
# would raise KeyError - which is exactly what the estimate gate produces.
SourcePosts = (pd.json_normalize(POST_ROWS).drop_duplicates("id") if POST_ROWS
             else pd.DataFrame(columns=["id", "creation_time", "text", "lang",
                                        "surface.id", "post_owner_id",
                                        "post_owner_type"]))

_capped = int(batch_report.get("capped", pd.Series(dtype=float)).fillna(False).sum())
_failed = int(batch_report.get("returned", pd.Series(dtype=float)).isna().sum())
_cap_used = CAP if IS_PREVIEW else JOB_RESULT_CAP

print(f"{len(SourcePosts):,} unique posts from {len(SourcePages):,} pages")
print(f"{len(POST_ROWS):,} raw rows -> {len(POST_ROWS) - len(SourcePosts):,} duplicates "
      f"dropped (windows overlap only at boundaries; a large number here means "
      f"JOB_WINDOWS are wrong)")
print(f"{_capped} of {len(batch_report)} jobs hit the {_cap_used:,} cap; "
      f"{_failed} failed")

if not IS_PREVIEW and len(POST_JOBS):
    _done = sum(1 for m in POST_JOBS.values() if str(m.get("status", "")).startswith("COMPLETE"))
    print(f"\njobs: {len(POST_JOBS)} submitted, {_done} complete, "
          f"{len(POST_JOBS) - _done} outstanding")
    print(f"rolling 7-day budget: {_budget_spent_7d():,} / {RECORD_BUDGET:,} spent")
    if len(batch_report) and "returned" in batch_report:
        _by_window = (batch_report.assign(
            window=batch_report["job_key"].str.split("|").str[1])
            .groupby("window")["returned"].agg(["size", "sum"]))
        print("\nposts by job window:")
        print(_by_window.to_string())

if _capped:
    print(f"\n!! {_capped} job(s) at the cap - those counts are FLOORS, not totals. "
          f"Narrow JOB_WINDOW and resubmit the affected keys.")
if SUBMIT_GATED:
    print("\n!! Estimate gate is closed - no posts fetched. Blocks below run on an "
          "EMPTY frame, which is handled but produces nothing.")
elif not ALL_JOBS_DONE:
    print("\n!! Jobs are still running - SourcePosts is PARTIAL. Re-run block 12 "
          "before trusting anything below.")


<!-- Last changed: Facebook_20260825D -->
## 4 — Term flag

`build_ORterms` sent single tokens to the API because there is no phrase search, so a
post containing "flu" anywhere qualifies. Block 14 re-imposes the real `PostText`
concepts — multi-word terms as adjacency patterns — and records the result as
`has_post_term` + `matched_terms`. **No rows are dropped.**

> **Fixed 2026-08-25.** The previous implementation was
> `hits.apply(lambda r: ", ".join(hits.columns[r]), axis=1)`, which raised
> *"Cannot set a DataFrame with multiple columns to the single column matched_terms"*.
> `DataFrame.apply(axis=1)` on a **zero-row** frame returns a DataFrame, not a Series —
> and a zero-row `SourcePosts` is exactly what the estimate gate produces. The numpy path
> below returns a plain list at every size, and is roughly two orders of magnitude
> faster across millions of rows, which matters now that the corpus is measured in
> millions rather than thousands.


In [ ]:
# BLOCK 14 - Term flag (requirement 1)
# Last changed: Facebook_20260825D
def term_re(term):
    """Multi-word terms become adjacency patterns; single tokens stay bounded tokens."""
    return re.compile(
        r"\b" + r"[\s\-/]+".join(map(re.escape, term.split())) + r"\b",
        re.IGNORECASE,
    )


TERMS = {t: term_re(t) for t in PostText}

_txt = (SourcePosts["text"].fillna("") if "text" in SourcePosts.columns
        else pd.Series("", index=SourcePosts.index, dtype=object))

hits = pd.DataFrame({t: _txt.str.contains(rx) for t, rx in TERMS.items()},
                    index=SourcePosts.index)

# --- the 2026-08-25 fix -----------------------------------------------------
# hits.apply(..., axis=1) returns a DATAFRAME when hits has zero rows, and
# pandas refuses to assign a multi-column frame to one column. Going through
# numpy returns a plain list at every size - including empty - and skips the
# per-row Python call for the ~99% of posts that match nothing.
_term_names = np.array(list(TERMS), dtype=object)
_hit_matrix = hits.to_numpy(dtype=bool) if len(hits.columns) else \
    np.zeros((len(SourcePosts), 0), dtype=bool)
_any_hit = _hit_matrix.any(axis=1) if _hit_matrix.size else \
    np.zeros(len(SourcePosts), dtype=bool)

matched = np.full(len(SourcePosts), "", dtype=object)
for _i in np.flatnonzero(_any_hit):
    matched[_i] = ", ".join(_term_names[_hit_matrix[_i]])

SourcePosts["has_post_term"] = _any_hit
SourcePosts["matched_terms"] = matched

# Kept only for continuity with the older notebooks - not used downstream.
PostsWithTerm = SourcePosts.loc[SourcePosts["has_post_term"]]

_rate = SourcePosts["has_post_term"].mean() if len(SourcePosts) else 0.0
print(f"{len(SourcePosts):,} posts kept; {int(_any_hit.sum()):,} ({_rate:.2%}) "
      f"match a PostText term")
if len(SourcePosts):
    print("\nper-term hits:")
    print(hits.sum().sort_values(ascending=False).to_string())
else:
    print("SourcePosts is empty - nothing to flag (see block 13).")


<!-- Last changed: Facebook_20260825A -->
## 5 — Page geography and category

`page_category` / `state` / `zip` describe the **posting Page**, not the author. Meta
exposes no author geography and no county, which is why the modeling grain for social
signal is state-or-coarser. Zip is included because it is the finest thing available,
not because it is a reliable locator of the person who wrote the post.


In [ ]:
# BLOCK 15 - Attach page geography and category (requirement 2)
# Last changed: Facebook_20260825D
PageLookup = (
    SourcePages[["id", "name", "page_category",
                 "location.state", "location.city", "location.zip"]]
    .rename(columns={"id": "page_id",
                     "name": "page_name",
                     "location.state": "state",
                     "location.city": "city",
                     "location.zip": "zip"})
)

SourcePosts = SourcePosts.merge(PageLookup, left_on="surface.id", right_on="page_id",
                            how="left")

# Grouping keys must not be NaN or the cells silently vanish from the aggregate.
SourcePosts["page_category"] = SourcePosts["page_category"].fillna("UNKNOWN")
SourcePosts["state"] = SourcePosts["state"].fillna("").replace("", "UNKNOWN")
SourcePosts["zip"] = (SourcePosts["zip"].fillna("").astype(str).str.strip()
                    .str.split("-").str[0].replace("", "UNKNOWN"))

SourcePosts["post_date"] = pd.to_datetime(
    SourcePosts["creation_time"], errors="coerce", utc=True
).dt.tz_convert(None).dt.date

_unmatched = SourcePosts["page_id"].isna().sum()
print(f"{_unmatched:,} posts did not join to a page "
      f"({_unmatched / max(len(SourcePosts), 1):.1%})")
print(f"{SourcePosts['post_date'].isna().sum():,} posts have an unparseable creation_time")
if len(SourcePosts):
    print(f"{(SourcePosts['zip'] == 'UNKNOWN').mean():.1%} of posts have no page zip")


<!-- Last changed: Facebook_20260825E -->
## 6 — Engagement, reliability flag, and sentiment

Post-level measures only. The decile bucketing that used to live here is gone — the
aggregate carries means, page-equal means, and extremes instead of twenty count
columns.


In [ ]:
# BLOCK 16 - Engagement, reliability, sentiment (requirements 3-6)
# Last changed: Facebook_20260825E
def _num(df, col):
    """Numeric column or a zero column if the API omitted it."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0.0, index=df.index)


SourcePosts["like_count"] = _num(SourcePosts, "statistics.like_count")
SourcePosts["sad_count"]  = _num(SourcePosts, "statistics.sad_count")
SourcePosts["like_sad"]   = SourcePosts["like_count"] + SourcePosts["sad_count"]

# --- requirement 4: reliability flag ---------------------------------------
SourcePosts["engagement_ge10"] = SourcePosts["like_sad"] >= ENGAGEMENT_MIN

# --- requirement 5: like share of like+sad ----------------------------------
# NaN where like+sad == 0: an undefined ratio, not a zero one.
SourcePosts["like_ratio"] = np.where(
    SourcePosts["like_sad"] > 0,
    SourcePosts["like_count"] / SourcePosts["like_sad"].replace(0, np.nan),
    np.nan,
)

# --- sad share, and the term-restricted view of it -------------------------
# sad_share is exactly 1 - like_ratio wherever the ratio is defined. It is
# carried separately because the signal of interest is a HIGH sad share, and
# restricting it to posts that matched a real PostText concept is what makes it
# a bird-flu signal rather than a general-misery signal.
SourcePosts["sad_share"] = np.where(
    SourcePosts["like_sad"] > 0,
    SourcePosts["sad_count"] / SourcePosts["like_sad"].replace(0, np.nan),
    np.nan,
)
SourcePosts["sad_share_term"] = SourcePosts["sad_share"].where(SourcePosts["has_post_term"])

# --- requirement 6: sentiment ----------------------------------------------
# The term-restricted view of sentiment, for the same reason sad_share_term
# exists: an extreme is only an H5N1 signal if the post was about H5N1.
SourcePosts["sentiment"] = score_sentiment(
    SourcePosts["text"] if "text" in SourcePosts.columns
    else pd.Series("", index=SourcePosts.index, dtype=object)
)

_n = max(len(SourcePosts), 1)
print(f"like+sad >= {ENGAGEMENT_MIN}: {int(SourcePosts['engagement_ge10'].sum()):,} "
      f"({SourcePosts['engagement_ge10'].sum() / _n:.1%}) of {len(SourcePosts):,} posts")
print(f"like_ratio defined for {int(SourcePosts['like_ratio'].notna().sum()):,} posts "
      f"(mean {SourcePosts['like_ratio'].mean():.3f})")
print(f"sentiment scored for {int(SourcePosts['sentiment'].notna().sum()):,} posts "
      f"via {SENTIMENT_ENGINE} (mean {SourcePosts['sentiment'].mean():.3f})")
SourcePosts["sentiment_term"] = SourcePosts["sentiment"].where(
    SourcePosts["has_post_term"])

print(f"sad_share defined for {int(SourcePosts['sad_share'].notna().sum()):,} posts; "
      f"{int(SourcePosts['sad_share_term'].notna().sum()):,} of those match a PostText term")


<!-- Last changed: Facebook_20260825C -->
## 7 — The aggregate table

One row per **(PageCategory, State, Zip, Date)**, built by `aggregate_posts`, which block
20 reuses at the state grain so the two tables cannot drift apart.

Every `n_*` column is a count of posts except `n_pages`, so `n_posts` is the denominator
for `n_posts_term` and `n_engagement_ge10`, and `n_ratio_defined + n_ratio_undefined ==
n_posts`.

The three ratio columns are three different weightings of the same reactions —
reaction-weighted (`like_ratio_total`), post-weighted (`mean_like_ratio`), and
page-weighted (`mean_like_ratio_page_eq`) — and `max_sad_share_term` is not a weighting
at all but the single loudest term-matching distress signal in the cell. See the intro
for why all four are carried.

All of them are NaN in a cell where nothing qualified — that is correct, and must not be
filled with 0. A NaN `max_sad_share_term` means "no term-matching post here had any
reactions", which is not the same as "no distress here".


In [ ]:
# BLOCK 17 - Aggregate table
# Last changed: Facebook_20260825E
GRP       = ["page_category", "state", "zip", "post_date"]
GRP_STATE = ["page_category", "state", "post_date"]


def aggregate_posts(df, keys):
    """Build the aggregate at any grain. Used for BOTH the zip-day and state-day
    tables, so the two can never drift apart.

    Three weightings of the same reactions live here on purpose:

      like_ratio_total         pooled over reactions. A national brand with
                               50,000 reactions dominates the cell.
      mean_like_ratio          mean over POSTS - each post counts once.
      mean_like_ratio_page_eq  mean over PAGES - each page counts once, however
                               big, so ten small farms do not collapse into one
                               national brand.

    max_sad_share_term is not an average: it is the loudest single distress
    signal in the cell among posts matching a real PostText concept. One small
    farm reporting a dead bird is the event of interest, and every mean above
    buries it by construction.
    """
    base = (
        df.groupby(keys, dropna=False)
        .agg(
            n_posts              = ("id", "size"),
            n_posts_term         = ("has_post_term", "sum"),
            n_engagement_ge10    = ("engagement_ge10", "sum"),
            n_ratio_defined      = ("like_ratio", "count"),
            n_sentiment_scored   = ("sentiment", "count"),
            n_term_ratio_defined = ("sad_share_term", "count"),
            n_term_sentiment_scored = ("sentiment_term", "count"),
            like_count_total     = ("like_count", "sum"),
            sad_count_total      = ("sad_count", "sum"),
            mean_like_ratio      = ("like_ratio", "mean"),
            mean_sentiment       = ("sentiment", "mean"),
            # No median. A median is a central-tendency statistic, and every
            # central-tendency statistic buries the single distressed small farm
            # this notebook exists to find. The extremes replace it.
            min_sentiment        = ("sentiment", "min"),
            min_sentiment_term   = ("sentiment_term", "min"),
            max_sad_share_term   = ("sad_share_term", "max"),
        )
    )

    # Page-equal weighting. Pooling WITHIN a page is deliberate - a page's own
    # ratio should use all of its reactions. It is only ACROSS pages that size
    # is ignored.
    per_page = (
        df.groupby(keys + ["page_id"], dropna=False)
        .agg(like=("like_count", "sum"), sad=("sad_count", "sum"),
             sent=("sentiment", "mean"))
    )
    _ls = per_page["like"] + per_page["sad"]
    per_page["ratio"] = np.where(_ls > 0, per_page["like"] / _ls.replace(0, np.nan),
                                 np.nan)

    page_eq = (
        per_page.groupby(level=list(range(len(keys))))
        .agg(n_pages                 = ("ratio", "size"),
             mean_like_ratio_page_eq = ("ratio", "mean"),
             mean_sentiment_page_eq  = ("sent", "mean"))
    )

    out = base.join(page_eq)
    out["n_ratio_undefined"] = out["n_posts"] - out["n_ratio_defined"]

    _tot = out["like_count_total"] + out["sad_count_total"]
    out["like_ratio_total"] = np.where(
        _tot > 0, out["like_count_total"] / _tot.replace(0, np.nan), np.nan)

    int_cols = ["n_posts", "n_pages", "n_posts_term", "n_engagement_ge10",
                "n_ratio_defined", "n_ratio_undefined", "n_sentiment_scored",
                "n_term_ratio_defined", "n_term_sentiment_scored",
                "like_count_total", "sad_count_total"]
    float_cols = ["like_ratio_total", "mean_like_ratio", "mean_like_ratio_page_eq",
                  "mean_sentiment", "mean_sentiment_page_eq",
                  "min_sentiment", "min_sentiment_term", "max_sad_share_term"]

    # Model-ready companions. The measurement columns above stay NaN-honest -
    # NaN means "not measurable here", which is true and must stay sayable. The
    # *_model columns are what the H5N1 model consumes, so it never has to drop
    # a row. Which rows were filled is fully recoverable from the n_term_*
    # counts, so nothing is lost by having both.
    model_cols = ["max_sad_share_term_model", "min_sentiment_term_model"]
    out["max_sad_share_term_model"] = out["max_sad_share_term"].fillna(
        MODEL_FILL_VALUE)
    out["min_sentiment_term_model"] = out["min_sentiment_term"].fillna(
        MODEL_FILL_VALUE)

    out[int_cols] = out[int_cols].fillna(0).astype(int)

    return (out[int_cols + float_cols + model_cols].reset_index()
            .sort_values(keys).reset_index(drop=True))


AggByCatStateZipDate = aggregate_posts(SourcePosts, GRP)

print(f"{len(AggByCatStateZipDate):,} aggregate rows covering "
      f"{AggByCatStateZipDate['n_posts'].sum():,} posts from "
      f"{AggByCatStateZipDate['n_pages'].sum():,} page-cells")
if len(AggByCatStateZipDate):
    _a = AggByCatStateZipDate
    _n = len(_a)
    _thin = (_a["n_posts"] == 1).mean()
    print(f"{_thin:.1%} of cells hold a single post - RETAINED, see block 20")

    # Why max_sad_share_term is NaN, split by cause. The two causes mean
    # different things and it is worth knowing which one dominates.
    _have = int(_a["max_sad_share_term"].notna().sum())
    _no_term = int((_a["n_posts_term"] == 0).sum())
    _term_no_react = int(((_a["n_posts_term"] > 0)
                          & (_a["n_term_ratio_defined"] == 0)).sum())
    print(f"\nmax_sad_share_term measured in {_have:,} of {_n:,} cells "
          f"({_have / _n:.1%})")
    print(f"  NaN - no term-matching post at all      : {_no_term:,}")
    print(f"  NaN - term post(s) but zero reactions   : {_term_no_react:,}")
    print(f"  -> max_sad_share_term_model fills both with {MODEL_FILL_VALUE}; "
          f"n_posts_term and\n     n_term_ratio_defined recover which rows those "
          f"were.")

    _hs = int(_a["min_sentiment_term"].notna().sum())
    print(f"min_sentiment_term measured in {_hs:,} of {_n:,} cells "
          f"({_hs / _n:.1%}); min_sentiment_term_model has no NaN")
AggByCatStateZipDate.head(20)


<!-- Last changed: Facebook_20260825A -->
## 8 — Sanity checks

Run these before believing anything. Each one has failed at least once in a
comparable pipeline.


In [ ]:
# BLOCK 18 - Sanity checks
# Last changed: Facebook_20260825F
checks = []


def chk(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})


a = AggByCatStateZipDate

chk("aggregate post count == SourcePosts row count",
    a["n_posts"].sum() == len(SourcePosts),
    f"{a['n_posts'].sum():,} vs {len(SourcePosts):,}")

chk("n_posts_term <= n_posts", (a["n_posts_term"] <= a["n_posts"]).all())
chk("n_engagement_ge10 <= n_posts", (a["n_engagement_ge10"] <= a["n_posts"]).all())
chk("n_sentiment_scored <= n_posts", (a["n_sentiment_scored"] <= a["n_posts"]).all())

chk("ratio defined + undefined == n_posts",
    (a["n_ratio_defined"] + a["n_ratio_undefined"] == a["n_posts"]).all())

chk("mean_like_ratio in [0, 1] where defined",
    a["mean_like_ratio"].dropna().between(0.0, 1.0).all())
chk("mean_like_ratio NaN exactly where n_ratio_defined == 0",
    (a["mean_like_ratio"].isna() == (a["n_ratio_defined"] == 0)).all())

chk("mean_sentiment in [-1, 1] where defined",
    a["mean_sentiment"].dropna().between(-1.0, 1.0).all())
chk("mean_sentiment NaN exactly where n_sentiment_scored == 0",
    (a["mean_sentiment"].isna() == (a["n_sentiment_scored"] == 0)).all())

chk("no NaN grouping keys", not a[GRP].isna().any().any())

# --- the 20260825C statistics ----------------------------------------------
chk("n_pages >= 1 wherever there are posts",
    (a.loc[a["n_posts"] > 0, "n_pages"] >= 1).all())
chk("n_pages <= n_posts", (a["n_pages"] <= a["n_posts"]).all(),
    "a page cannot contribute more cells than it has posts")
chk("n_term_ratio_defined <= n_posts_term",
    (a["n_term_ratio_defined"] <= a["n_posts_term"]).all())
chk("like_ratio_total in [0, 1] where defined",
    a["like_ratio_total"].dropna().between(0.0, 1.0).all())
chk("mean_like_ratio_page_eq in [0, 1] where defined",
    a["mean_like_ratio_page_eq"].dropna().between(0.0, 1.0).all())
chk("page-equal ratio NaN exactly where no post had reactions",
    (a["mean_like_ratio_page_eq"].isna() == (a["n_ratio_defined"] == 0)).all())
chk("page-equal sentiment NaN exactly where nothing scored",
    (a["mean_sentiment_page_eq"].isna() == (a["n_sentiment_scored"] == 0)).all())
chk("mean_sentiment_page_eq in [-1, 1] where defined",
    a["mean_sentiment_page_eq"].dropna().between(-1.0, 1.0).all())
chk("max_sad_share_term in [0, 1] where defined",
    a["max_sad_share_term"].dropna().between(0.0, 1.0).all())
chk("max_sad_share_term NaN exactly where no term post had reactions",
    (a["max_sad_share_term"].isna() == (a["n_term_ratio_defined"] == 0)).all())

# --- the 20260825E statistics ----------------------------------------------
chk("n_term_sentiment_scored <= n_posts_term",
    (a["n_term_sentiment_scored"] <= a["n_posts_term"]).all())
chk("min_sentiment in [-1, 1] where defined",
    a["min_sentiment"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment_term in [-1, 1] where defined",
    a["min_sentiment_term"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment <= mean_sentiment wherever both are defined",
    (a["min_sentiment"] <= a["mean_sentiment"] + 1e-12).fillna(True).all())
chk("min_sentiment_term NaN exactly where no term post scored",
    (a["min_sentiment_term"].isna() == (a["n_term_sentiment_scored"] == 0)).all())
chk("median_sentiment is gone", "median_sentiment" not in a.columns,
    "a median buries the single distressed small farm - that is why it was cut")

# --- model-ready companions -------------------------------------------------
chk("model columns contain no NaN",
    not a[["max_sad_share_term_model", "min_sentiment_term_model"]].isna().any().any(),
    "the H5N1 model must never have to drop a row")
chk("model columns equal the measurement wherever it is defined",
    (a.loc[a["max_sad_share_term"].notna(), "max_sad_share_term_model"]
     .equals(a.loc[a["max_sad_share_term"].notna(), "max_sad_share_term"]))
    and (a.loc[a["min_sentiment_term"].notna(), "min_sentiment_term_model"]
         .equals(a.loc[a["min_sentiment_term"].notna(), "min_sentiment_term"])),
    "a fill must not alter a measured value")
chk("filled rows are recoverable from the counts",
    (a["max_sad_share_term"].isna() == (a["n_term_ratio_defined"] == 0)).all(),
    "n_term_ratio_defined == 0 is exactly the filled set")

# --- collection completeness -----------------------------------------------
chk("no job failed", _failed == 0, f"{_failed} failed jobs")
chk("no job hit the result cap", _capped == 0,
    f"{_capped} capped at {_cap_used:,} - counts are floors, not totals")
chk("all async jobs finished", ALL_JOBS_DONE,
    "jobs still running - SourcePosts is partial" if not ALL_JOBS_DONE else "")
chk("every planned job was submitted",
    IS_PREVIEW or SUBMIT_GATED or not len(_job_plan(SourcePages["id"])) or
    all(j["job_key"] in POST_JOBS for j in _job_plan(SourcePages["id"])),
    "budget or MAX_JOBS_PER_RUN stopped submission - collection is STAGED, "
    "re-run block 12 next window")

chk("post_date inside the requested window",
    (pd.Series(pd.to_datetime(a["post_date"], errors="coerce")).between(
        pd.Timestamp(RUN_START), pd.Timestamp(RUN_END)) | a["post_date"].isna()).all(),
    f"{RUN_START} .. {RUN_END} ({RUN_MODE})")

# Page discovery is sync-only in both modes, so slices can be silently truncated.
_n_capped_slices = int(((page_slices["capped"] == True)        # noqa: E712
                        & (~page_slices["sub_split"])).sum()) if len(page_slices) else 0
_n_failed_slices = int(page_slices["capped"].isna().sum()) if len(page_slices) else 0
chk("no page-discovery slice hit the sync cap", _n_capped_slices == 0,
    f"{_n_capped_slices} of {len(page_slices)} slices capped at "
    f"{SYNC_SEARCH_CAP:,} - the page universe is a sample there, not a census "
    f"(13 on 2026-08-24)")
chk("no page-discovery slice failed", _n_failed_slices == 0,
    f"{_n_failed_slices} slice(s) errored and contributed ZERO pages - re-run "
    f"block 07 with RETRY_FAILED_SLICES=True before trusting this shard")
chk("discovery ran across all production categories",
    IS_SMOKE or DISCOVER_ALL_CATEGORIES,
    "shard-only discovery finds fewer of the shard's OWN pages, because capped "
    "slices mean more slices find more unique pages")

chk("RUN_MODE is 'full'", RUN_MODE == "full",
    "preview output is a smoke test, not a dataset")

# --- shard integrity --------------------------------------------------------
chk("aggregate holds only this shard's categories",
    set(a["page_category"]) <= set(PageCategories) | {"UNKNOWN"},
    f"{sorted(set(a['page_category']) - set(PageCategories) - {'UNKNOWN'})}")
chk("shard is a production shard", not IS_SMOKE,
    "SHARD='smoke' is an environment check on Zoo, not a dataset")
chk("no UNKNOWN page_category",
    "UNKNOWN" not in set(a["page_category"]),
    "posts that did not join to a page cannot be attributed to a shard")
chk("JOB_WINDOW set in full mode", IS_PREVIEW or JOB_WINDOW is not None,
    "one job per batch over the whole window averaged ~115,000 posts on the "
    "2026-08-24 estimate, against a 100,000 cap")

report = pd.DataFrame(checks)
print(f"{int(report['pass'].sum())} of {len(report)} checks passed\n")
print(report.to_string(index=False))


In [ ]:
# BLOCK 19 - Distribution report
# Last changed: Facebook_20260825A
# Where the mass actually sits - useful for deciding whether zip is a usable grain.
print("posts per aggregate cell:")
print(AggByCatStateZipDate["n_posts"].describe().round(2).to_string())
if len(AggByCatStateZipDate):
    print(f"\ncells with a single post: "
          f"{(AggByCatStateZipDate['n_posts'] == 1).mean():.1%}")
    print("\ntop categories by post volume:")
    print(AggByCatStateZipDate.groupby("page_category")["n_posts"]
          .sum().sort_values(ascending=False).to_string())
    print("\ntop states by post volume:")
    print(AggByCatStateZipDate.groupby("state")["n_posts"]
          .sum().sort_values(ascending=False).head(15).to_string())


<!-- Last changed: Facebook_20260825D -->
## 9 — Export shape

⚠️ **Nothing leaves the enclave without Meta's output review.** Block 20 only *builds*
the reviewable object; it deliberately does not write a file. It also refuses to treat a
preview run or an unfinished async pull as an exportable result.

`MIN_CELL_POSTS` suppresses thin cells and **defaults to 0 — retain everything**. That
is a considered position: a single-post cell is very often the signal rather than noise,
and suppressing at 5 or 10 deletes exactly the small single-farm observations
`max_sad_share_term` exists to surface. If a reviewer requires suppression the lever is
there, and the block reports how many early-warning cells the threshold would destroy
before you accept it.

**I am confabulating** on whether Meta's output review requires small-cell suppression
for an export of counts and ratios carrying no post text. Ask them rather than inferring
it; nothing inside the enclave needs it either way.

The state rollup is rebuilt from `SourcePosts` through the same `aggregate_posts` helper,
not summed out of the zip table. Summing recovers the totals but not the page-equal means
or the sentiment columns — which is exactly why the previous version had to drop
sentiment from the rollup. One code path, two grains, no drift.

If output review clears the aggregate, note that `job.write_data_to_file(directory=...,
filename=...)` writes **raw post rows**, not this table — a within-enclave convenience,
not an export route. The same is true of every checkpoint file.


In [ ]:
# BLOCK 20 - Export shape
# Last changed: Facebook_20260825D
if IS_PREVIEW:
    print("RUN_MODE='preview' - this is a smoke test, not an exportable result.\n"
          "Set RUN_MODE='full', rerun the notebook, then build the export.\n")
elif SUBMIT_GATED:
    print("The estimate gate stopped submission - there is nothing to export.\n")
elif not ALL_JOBS_DONE:
    print("Async jobs are still running - this aggregate is PARTIAL.\n"
          "Re-run block 12 until every job is COMPLETE before exporting.\n")

ExportAgg = AggByCatStateZipDate.copy()

if MIN_CELL_POSTS > 0:
    _keep = ExportAgg["n_posts"] >= MIN_CELL_POSTS
    _lost_sig = int(ExportAgg.loc[~_keep, "max_sad_share_term"].notna().sum())
    print(f"suppressing {int((~_keep).sum()):,} of {len(ExportAgg):,} cells "
          f"below n_posts={MIN_CELL_POSTS} "
          f"({int(ExportAgg.loc[~_keep, 'n_posts'].sum()):,} posts)")
    if _lost_sig:
        print(f"  !! {_lost_sig:,} of the suppressed cells carry a "
              f"max_sad_share_term - that is the early-warning signal being "
              f"deleted. Confirm review actually requires this.")
    ExportAgg = ExportAgg[_keep].reset_index(drop=True)
else:
    # Deliberate, not an unset knob: a single-post cell is very often the signal
    # rather than noise, and suppressing at 5 or 10 deletes exactly the small
    # single-farm observations max_sad_share_term exists to surface.
    _thin = int((ExportAgg["n_posts"] == 1).sum())
    print(f"MIN_CELL_POSTS=0 - thin cells RETAINED by design "
          f"({_thin:,} single-post cells kept).")
    print("  Set MIN_CELL_POSTS only if Meta's output review requires it; nothing "
          "inside\n  the enclave needs it, and it costs signal.")

# State-level fallback if zip-day is judged too granular by output review.
# Rebuilt from SourcePosts through the SAME aggregate_posts helper rather than
# summed out of the zip table. Summing could recover the totals, but neither the
# page-equal means nor the sentiment columns survive that route - which is why
# the previous version had to drop sentiment from the rollup entirely.
ExportAggState = aggregate_posts(SourcePosts, GRP_STATE)

print(f"\nExportAgg      : {len(ExportAgg):,} rows x {ExportAgg.shape[1]} cols")
print(f"ExportAggState : {len(ExportAggState):,} rows x {ExportAggState.shape[1]} cols "
      f"(same columns as the zip grain, sentiment included)")

print(f"\nrun_mode={RUN_MODE}  window={RUN_START}..{RUN_END}  "
      f"engine={SENTIMENT_ENGINE}  query_mode={POST_QUERY_MODE}  "
      f"farm_only={RESTRICT_TO_FARM_ONLY}  engagement_min={ENGAGEMENT_MIN}")
if not IS_PREVIEW:
    print(f"job_mode={JOB_MODE}  job_window={JOB_WINDOW}  jobs={len(POST_JOBS)}  "
          f"all_complete={ALL_JOBS_DONE}")

# ---------------------------------------------------------------------------
# Write this shard's contribution. CSV as well as pickle: a pickle cannot pass
# output review, so if researchers turn out NOT to share a filesystem the CSV is
# the artifact each person carries out and block 22 recombines outside.
# ---------------------------------------------------------------------------
os.makedirs(SHARD_DIR, exist_ok=True)

ExportAgg = ExportAgg.assign(shard=SHARD)
ExportAggState = ExportAggState.assign(shard=SHARD)

_shard_manifest = dict(_ckpt_manifest("shard-export"),
                       categories=PageCategories,
                       min_cell_posts=MIN_CELL_POSTS,
                       n_rows=len(ExportAgg))

for _name, _obj in (("exportagg", ExportAgg), ("exportaggstate", ExportAggState)):
    _stem = os.path.join(SHARD_DIR, f"shard_{SHARD}_{_name}")
    _obj.to_csv(_stem + ".csv", index=False)
    _obj.to_pickle(_stem + ".pkl.gz", compression="gzip")
with open(os.path.join(SHARD_DIR, f"shard_{SHARD}_manifest.json"), "w",
          encoding="utf-8") as _fh:
    json.dump(_shard_manifest, _fh, default=str, indent=2)

print(f"\nshard '{SHARD}' written to {SHARD_DIR}")
for _f in sorted(f for f in os.listdir(SHARD_DIR) if f.startswith(f"shard_{SHARD}_")):
    print(f"  {_f}")
if IS_SMOKE:
    print("\n!! SMOKE shard - delete these before combining anything real.")

ExportAgg.head(20)


<!-- Last changed: Facebook_20260825A -->
## 10 — Final checkpoint

Persist the derived frames too. They are all recomputable from `POST_ROWS`, so this is
convenience rather than insurance — but it is the difference between reopening the
notebook tomorrow and re-running blocks 12–19 first.

Same boundary as everywhere else: this writes inside the enclave. `ExportAgg` becoming a
file here does **not** make it exported.


In [ ]:
# BLOCK 21 - Final checkpoint
# Last changed: Facebook_20260825B
checkpoint("final")

print(f"\nrolling 7-day budget: {_budget_spent_7d():,} / {RECORD_BUDGET:,} spent, "
      f"{_budget_remaining():,} available")

if not IS_PREVIEW:
    _plan_n = len(_job_plan(SourcePages["id"]))
    _left = _plan_n - len(POST_JOBS)
    if _left > 0:
        print(f"{_left:,} of {_plan_n:,} planned job(s) still unsubmitted - this "
              f"collection is STAGED.\nRe-run block 12 when the rolling window frees "
              f"up; the ledger and job ids are checkpointed.")

print(f"\nshard '{SHARD}' -> {CKPT_DIR}")
print("\nto resume in a fresh session: set the same SHARD, then run blocks\n"
      "00-04 with\n"
      "RESTORE_FROM_CHECKPOINT = True, then run down the notebook. Page discovery\n"
      "skips, submitted jobs are not resubmitted, polling reattaches, and the\n"
      "budget ledger carries its rolling window forward.")


<!-- Last changed: Facebook_20260825B -->
## 11 — Combining the shards

Run this once all three shards exist. It works in either arrangement:

* **shared enclave workspace** — everyone writes into the same `SHARD_DIR`, combine
  inside, and put the merged aggregate through output review once;
* **isolated workspaces** — each researcher takes their own `shard_<name>_exportagg.csv`
  through review, and the three approved CSVs are combined outside the enclave.

The block prefers CSV over pickle for exactly that reason: a pickle cannot pass review.

The concat is the easy half. What this block is really for is refusing to merge shards
that were not produced the same way — mismatched sentiment engine, query mode, date
window, engagement threshold, job window, or suppression threshold. Every one of those
produces a combined table that looks entirely healthy and is quietly wrong.


In [ ]:
# BLOCK 22 - Combine shards
# Last changed: Facebook_20260825C
COMBINE_SHARDS = ["nick", "max", "waree"]   # "smoke" is deliberately not here

# Fields that MUST agree across shards or the combined table is not one dataset.
_LOCKED = ("sentiment_engine", "query_mode", "run_start", "run_end",
           "engagement_min", "job_window", "min_cell_posts")


def _load_shard(name, which="exportagg"):
    """CSV first - a pickle cannot pass output review, so CSV is the portable one."""
    stem = os.path.join(SHARD_DIR, f"shard_{name}_{which}")
    if os.path.exists(stem + ".csv"):
        # dtype=str on the key columns is NOT optional: read_csv infers "05010"
        # as the integer 5010, silently renumbering every zip in CT/MA/ME/NH/
        # NJ/RI/VT - and a renumbered key will not match its own shard's rows.
        df = pd.read_csv(stem + ".csv",
                         dtype={"zip": str, "state": str,
                                "page_category": str, "shard": str})
        if "post_date" in df.columns:
            df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce").dt.date
        if "zip" in df.columns:
            df["zip"] = df["zip"].str.strip()
        return df, "csv"
    if os.path.exists(stem + ".pkl.gz"):
        return pd.read_pickle(stem + ".pkl.gz", compression="gzip"), "pkl"
    raise FileNotFoundError(f"no {which} for shard '{name}' in {SHARD_DIR}")


# What is actually on disk, so a missing shard reads as "not produced yet"
# rather than as a failure. A smoke shard is never combined - it is an
# environment check on Zoo, and combining it would put test data in the dataset.
_present = sorted({f.split("_")[1] for f in os.listdir(SHARD_DIR)
                   if f.startswith("shard_")}) if os.path.isdir(SHARD_DIR) else []
print(f"SHARD_DIR      : {SHARD_DIR}")
print(f"shards on disk : {_present or 'none'}")
print(f"shards wanted  : {COMBINE_SHARDS}")
if "smoke" in _present:
    print("  note: 'smoke' is the Zoo environment check and is never combined.")
_waiting = [n for n in COMBINE_SHARDS if n not in _present]
if _waiting:
    print(f"  waiting on    : {_waiting}  (not yet produced)")
print()

frames, manifests, problems = {}, {}, []

for _name in COMBINE_SHARDS:
    try:
        frames[_name], _fmt = _load_shard(_name)
        with open(os.path.join(SHARD_DIR, f"shard_{_name}_manifest.json"),
                  encoding="utf-8") as _fh:
            manifests[_name] = json.load(_fh)
        print(f"  {_name:<8} {len(frames[_name]):>8,} rows  ({_fmt})  "
              f"engine={manifests[_name].get('sentiment_engine')}")
    except Exception as e:
        problems.append(f"shard '{_name}' has not been produced yet")
        print(f"  {_name:<8} not produced yet")

# --- the config lock --------------------------------------------------------
print("\nconfig lock:")
for _f in _LOCKED:
    vals = {n: m.get(_f) for n, m in manifests.items()}
    agree = len(set(map(str, vals.values()))) <= 1
    print(f"  {_f:<18} {'OK  ' if agree else 'DIFF'}  {vals}")
    if not agree:
        problems.append(f"{_f} differs across shards: {vals}")

# --- disjointness -----------------------------------------------------------
cats = {n: set(df['page_category']) for n, df in frames.items()}
for _a in cats:
    for _b in cats:
        if _a < _b and cats[_a] & cats[_b]:
            problems.append(f"shards {_a}/{_b} share categories {cats[_a] & cats[_b]}")

if problems:
    print("\n!! NOT COMBINING - resolve these first:")
    for _p in problems:
        print(f"  - {_p}")
    CombinedAgg = None
else:
    GRP_C = ["page_category", "state", "zip", "post_date"]
    CombinedAgg = (pd.concat(frames.values(), ignore_index=True)
                   .sort_values(GRP_C).reset_index(drop=True))

    dup = CombinedAgg.duplicated(subset=GRP_C).sum()
    assert dup == 0, f"{dup} duplicate aggregate cells - shards are not disjoint"

    print(f"\nCombinedAgg: {len(CombinedAgg):,} rows, "
          f"{CombinedAgg['n_posts'].sum():,} posts")
    print(CombinedAgg.groupby("shard")
          .agg(rows=("n_posts", "size"), posts=("n_posts", "sum"),
               categories=("page_category", "nunique")).to_string())
    print(f"\nstates {CombinedAgg['state'].nunique()}   "
          f"dates {CombinedAgg['post_date'].nunique()}   "
          f"categories {CombinedAgg['page_category'].nunique()}")
    CombinedAgg.head(20)
